# bridge
> Live browser–kernel messaging for dialog-native controls.

`bridge` is a narrow foundation for making the browser and kernel one continuous interactive surface. It carries semantic JSON messages between live DOM and live Python state; it does not itself define widgets, durable dialog projection, or context policy.

The larger goal is dialog-native controls: human interaction changes meaningful state, and a separate persistence layer projects that state into durable dialog artifacts that humans and models can later read.

```text
dialog source/output  = durable context
kernel values         = live working state
browser DOM           = live presentation and input surface
bridge + Channel      = bidirectional semantic transport
```

In [ ]:
#| default_exp bridge

In [ ]:
import asyncio
from fastcore.foundation import AttrDict as AD
from fastcore.parallel import bg_task
from dialoghelper.core import Channel
from pote.basic import simple_id, id_gen
from pote.dutil import find_var

In [ ]:
#| hide
import asyncio, regex, json, os, inspect
from collections import defaultdict
from importlib import resources
from IPython.display import HTML, Markdown
from fastcore.meta import delegates
from fastcore.net import maybe_await
from fastcore.test import *
from fastcore.parallel import bg_task
from fastcore.xml import to_xml
from fasthtml.core import FastHTML
from fasthtml.xtend import Script
from pyskills import *
from pyskills.core import allow, doc
from fasthtml.components import Div, Span, Button, Input, Label, Output, Select, Textarea, Option
from dialoghelper.core import add_msg, iife_a, add_html_a, read_msg, add_scr_a, js_run_a, js_eval_a, dialog_link, update_msg, find_msgs, js_eval, js_run, add_mod_a
from pote.dutil import find_var, link_msg, wait_for_a
from pote.server import stop_server_a, setup_mod, add_js_msg
from htmx_bridge.core import get_hx

In [ ]:
# from pyskills.skill import *
# from dialoghelper.solveitskill import *
# from fastcore.editskill import *
# from nbdev.skill import *
# from aidialog.dlgskill import *

In [ ]:
_new_id = id_gen()

In [ ]:
hx = get_hx()

In [ ]:
await stop_server_a()
app=rt=p=None

## earlier exploration

(archived at <a href="/dialog_?name=prj%2Fpote%2Fexplorer%2F04_bridge_Channel_checkpoint" target="_blank">prj/pote/explorer/04_bridge_Channel_checkpoint</a> exploration about messaging transport using `Channel`.)

The exploration has established the essential architecture.

> A reusable browser bridge module plus a declarative `hx-sync` plugin that maintains bidirectional live synchronization over `Channel` for standard form controls and dotted kernel paths.

The earlier control/output experiment also proved that dialog message outputs can hold durable rendered state. That should remain a separate persistence layer, rather than being mixed into the transport and sync mechanics now.

That milestone explicitly excludes source/output persistence, AST patching, `idiomorph`, reconnect policy, multi-peer identity, nested controls, and arbitrary object-path semantics.

## decisions

**Bridge invariants**

1. **The dialog is the eventual durable record.** Browser DOM and kernel objects are live projections, not a second persistent widget-state store.

2. **This notebook implements live synchronization only.** Durable projection into note, prompt, code source, code output, or raw messages belongs to a later layer.

3. **`Channel` is the active transport.** It carries opaque JSON frames and does not impose widget identity, routing policy, or persistence.

4. **The browser bridge is generic; plugins own semantics.** `bridge` owns browser-local eventing, plugin lifecycle, and Channel transport; `hx-sync`, HTMX lifecycle adaptation, controls, graphs, and other domain behavior must not leak into generic transport code.

5. **`hx-sync` is declarative.** Users and agents state a path and render a supported control; they do not manually install listeners or send Channel frames. `hx-sync="path"` binds a supported form-control value to a dotted kernel path.

6. **One kernel dispatcher consumes Channel events.** Semantic handlers are registered by `key`; independent consumers would race for frames.

7. **No automatic DOM-mutation policy yet.** Delegated `input` and `change` events work across replacements, but an HTMX swap that introduces a new value does not currently update the kernel unless it produces an event.

Durable source/output projection and HTMX lifecycle adaptation are separate layers.

## long term vision (large)

I need to explain why I want more interactive "controls" in the dialog. Solveit is all about editing the context sent upstream to the LLMs. That's the core, the mission, the origin of Solveit, the non-linear editing of conversation history. Everything else are implementation details. But those details are important and constraint a lot what, when and how can be edited. A dialog contains four editable artifacts: note messages (markdown, now with the new flavor `mdhtml`, more powerful), code (input + output) with persistent kernel, prompt (just markdown iunder the hood), raw (transparent text). Code outputs can of course have anything iPython can generate. Prompt outputs are just markdown. In the two cases, the LLM only sees what's is originally generated (or with prompts, is manually edited).

I want to have more powerful editing features to improve the context the model see. I want the model generating a list of clickable options instead of markdown list **and** see in the next turn what the user clicks. I want to interact with a graph using widgets and the model see the results, not an image capture.

For that we need a realiable way to connect live DOM with dialog source (input and outputs). And I want all that to be actianable in a transparent way that allows agents to state what, no how (thus the declarative exploration).

The controls must be composable arbitrarily, fully expressive. I want simple buttons and complex tree views that interact with the dialog. The controls/widgets must be fully usable by humans and AIs.

The product goal is not “widgets in a notebook.” It is **interactive, durable context editing**.

A dialog-native control should let a human manipulate a live interface while ensuring that the resulting state becomes part of the dialog the model reads on its next turn. The browser is therefore not a private state store. It is a view and an input surface over dialog-backed state.

The core invariant should be:

```text
human or agent interaction
→ semantic state change
→ durable dialog update
→ live DOM reflects that update
→ later LLM context reflects that update
```

For example, a model might generate a choice control rather than a markdown list. When the user picks an option, the next model turn should see something equivalent to:

```markdown
Selected deployment target: production
```

—not merely an unchanged list whose clicked state existed briefly in a browser tab.

For a graph, the model should see semantic results such as:

```text
Selected nodes: auth, billing
Expanded subgraph: auth → token → session
Layout/filter: dependency view, external services hidden
```

The rendered graph is useful for the human, but a screenshot is not the durable interaction record. The durable representation is the graph’s meaningful state.

This implies three related but distinct layers:

1. **Live binding**  
   DOM controls and kernel values synchronize while the dialog is open.

   ```html
   <input hx-sync="a.b.c">
   ```

   is a useful primitive here.

2. **Durable dialog projection**  
   A meaningful state change patches a note, prompt, raw message, code source, code output, or a deliberate combination of these. This is what survives save, reload, browser replacement, and kernel restart.

3. **Context projection**  
   The durable representation must be legible to the model. It may be rendered HTML/`mdhtml`, structured text, a compact JSON-like state block, or executable code plus a persisted output—whichever makes the interaction’s result clear in context.

The important distinction is:

```text
DOM value        = transient presentation state
kernel object    = transient working state
dialog artifact  = durable truth
LLM context      = view of durable truth
```

A useful definition of a control, then, is not “an element with event listeners.” It is a **composable, dialog-backed semantic unit** with:

- a stable identity;
- a value, possibly recursively structured;
- a human-facing rendering;
- an AI-readable durable representation;
- declared interactions;
- a persistence rule;
- optional children/composition.

A simple button might have no interesting persistent value beyond an event record:

```text
Button: “Approve plan”
State: approved
```

A selector has a scalar value:

```text
Model choice: Claude / GPT / Gemini
Selected: GPT
```

A tree view has structured state:

```text
Expanded: [/src, /src/pote]
Selected: /src/pote/bridge.py
```

A graph can have a richer value:

```python
dict(
    selected=['auth', 'billing'],
    expanded=['auth'],
    filters=dict(kind=['service'], external=False),
    layout='dependency',
)
```

All of these should compose. A form is a control whose value is a dictionary of child-control values. A graph explorer may contain filters, selection state, a details panel, and actions. A notebook-like interactive analysis can contain tables, sliders, tree views, and buttons without each part inventing its own persistence system.

That points to the declarative contract agents should eventually use:

```text
State what the control means,
what value it exposes,
what interactions it permits,
and what durable dialog artifact should reflect the result.

Do not require the agent to specify websockets, listeners, DOM mutation observers,
Channel frames, or source-patching mechanics.
```

For example, an agent should be able to express something conceptually like:

```text
Render a deployment selector bound to `plan.target`.
Persist the selected target in this note as “Target: <value>”.
Expose the selected value in the next model context.
```

The runtime chooses the mechanics: live DOM binding, kernel synchronization, output update, source patch, and any necessary browser refresh.

This also clarifies why arbitrary composition matters. The primitive is not a catalog of predefined widgets; it is a general relationship:

```text
rendered structure ↔ semantic value ↔ durable dialog representation
```

Buttons, forms, trees, tables, graphs, editors, and custom domain controls are different renderings of that relationship.

The next implementation phase should therefore be driven by a product-level target:

> Build a dialog-native control model in which every meaningful interaction has a durable, model-readable representation in the dialog, while the browser provides a rich live interface over that representation.

`hx-sync` and the Channel bridge are enabling primitives. They are not the product contract on their own.

In that view, messages (code, note, prompt) **are** controls, widgets that compose other widgets: markdown editor, code editor, editing buttons, run buttons, etc. We need to expand that control catalog and capabilities.

Unfortunately, though Solveit is very well thought of and has a rich API, there's a lot of aspects we can't control and will clash with our vision. But it is what it is and we have to work with that. In the long term I'll probably implmenet my own Solveit. For now, Solve is the substrate where we'll implement a better context editor.

Yes. The unit of composition is not merely an added slider or graph; it is the **message itself**.

A code message is already a compound control:

```text
source editor + output view + run action + execution state + output persistence
```

A note or prompt message is similarly a compound control:

```text
editable durable text + rendered view + context contribution + editing actions
```

The work is therefore to extend the existing message-control catalog, rather than bolt a separate widget system beside it.

The target architecture is:

```text
Solveit message substrate
  └─ durable source / output / metadata
      └─ message-native controls
          └─ composable domain controls
              └─ durable, model-readable interaction state
```

A control should be able to live:

- **inside a message** — a choice list, form, tree, graph, structured editor;
- **around a message** — run controls, context inclusion controls, collapse/pin/branch/history tools;
- **across messages** — a control that selects, rewrites, links, groups, compares, or reorganizes dialog artifacts.

That last category is especially important to Solveit’s mission. A context editor needs controls for operations such as:

```text
select these messages for the next prompt
turn these exploratory outputs into a compact summary
choose one branch of several alternatives
mark this result as durable context
replace a verbose trace with its extracted decisions
link a graph node to the message that produced it
```

These are not incidental UI enhancements. They are direct tools for editing the LLM’s working context.

The practical constraint suggests a layered strategy:

1. **Treat Solveit as the durable document and execution substrate.**  
   Respect its message types, persistence rules, rendering model, kernel, and APIs.

2. **Build an independent control layer above it.**  
   Controls express semantic state, composition, interaction, and durable projections without depending on Solveit internals more than necessary.

3. **Use adapters at the boundary.**  
   The bridge, Channel, DOM hooks, message-update APIs, and output/source patching are substrate adapters. They should be isolated so unavoidable Solveit constraints do not infect the control model.

4. **Prefer durable message projections over browser-only state.**  
   A control’s state must ultimately become source, output, metadata, or explicit context text in a message.

5. **Keep the future escape route open.**  
   If the control model depends on a small abstract document/message interface rather than Solveit-specific details, it can later move to a replacement host without redesigning buttons, trees, graphs, or context-editing workflows.

This reframes the implementation objective:

> Build a composable control system for editing durable conversational context, with Solveit as its first host rather than its final definition.

The browser bridge and `hx-sync` work are foundational because they establish one missing capability: live interaction can now reach meaningful kernel state. The next architectural problem is defining how a control declares its **durable projection into messages**—the part an AI and a reopened dialog can both reliably see.

## long term vision

The product goal is **interactive, durable context editing**, not merely widgets in a notebook.

Solveit’s purpose is to edit the context supplied to an LLM. Its messages—notes, prompts, code inputs and code outputs—are already durable, composable controls. The work is to extend that control catalog so humans can manipulate richer live interfaces while the resulting semantic state becomes part of the dialog that later humans and models read.

```text
human or agent interaction
→ semantic state change
→ durable dialog update
→ live DOM reflects that update
→ later LLM context reflects that update
```

The browser must not become a private state store:

```text
browser DOM     = live presentation and input
kernel values   = live working state
dialog artifact = durable truth
LLM context     = view of durable truth
```

A choice control, for example, should persist `Selected deployment target: production`, not merely leave a selected browser option. A graph should persist selected nodes, filters, expanded subgraphs, and layout choices—not a screenshot.

This separates three layers:

1. **Live binding** synchronizes DOM controls and kernel values while a dialog is open. `hx-sync="a.b.c"` is one useful primitive.
2. **Durable projection** writes meaningful changes into a note, prompt, raw message, code source, code output, or a deliberate combination.
3. **Context projection** makes that durable state legible to the next model turn, through concise text, structured state, `mdhtml`, or executable code and persisted output.

A control is therefore a composable, dialog-backed semantic unit with:

- stable identity;
- a scalar or structured value;
- human-facing rendering;
- AI-readable durable representation;
- declared interactions;
- a persistence rule;
- optional child controls.

Forms compose child values into dictionaries. Trees, tables, graphs, editors, and message-management tools are richer renderings of the same relationship:

```text
rendered structure ↔ semantic value ↔ durable dialog representation
```

This includes controls *inside* messages, *around* messages, and *across* messages. The last category is central to context editing: selecting messages for a prompt, choosing branches, linking artifacts, extracting decisions from traces, or replacing exploratory output with a durable summary.

Agents should declare meaning rather than mechanics:

```text
Render a deployment selector bound to `plan.target`.
Persist the selected target in this note as `Target: <value>`.
Expose that value in the next model context.
```

The runtime handles listeners, Channel frames, DOM updates, kernel synchronization, and message projection.

Implementation should remain layered:

1. Treat Solveit as the current durable document and execution substrate.
2. Keep the control model independent of Solveit-specific details.
3. Isolate Channel, browser hooks, and message APIs behind adapters.
4. Prefer durable message projections over browser-only state.
5. Preserve an escape route to another host by depending on a small abstract document/message interface.

`bridge` and `hx-sync` establish live semantic transport. They are enabling primitives, not the final product contract. The next architectural task is a declarative model for each control’s durable projection into dialog messages.

## What is this dialog about?

This dialog is designing `bridge`: a narrow, reusable layer for live browser–kernel messaging in Solveit.

Its immediate scope is:

- transport opaque semantic JSON frames over `Channel`;
- provide browser-side lifecycle/event plumbing;
- support a declarative `hx-sync="path"` plugin that binds standard form controls to dotted paths in live kernel state;
- keep generic transport separate from control-specific semantics.

The broader product goal is **dialog-native, durable context editing**. Browser interactions should not remain transient UI state: meaningful selections, form values, graph state, and actions should ultimately be projected into durable dialog messages so both a reopened dialog and the next LLM turn can read them.

The intended flow is:

```text
human or agent interaction
→ semantic state change
→ durable dialog update
→ live DOM reflects it
→ later LLM context reflects it
```

The notebook deliberately does **not** yet implement durable projection, source/output patching, reconnect policy, multi-peer identity, nested controls, or general object-path semantics. Those are later layers built on top of the bridge.

## browser bridge v0
Defines the generic browser–kernel messaging foundation for live, dialog-native interaction.

The browser bridge is deliberately generic.

It owns:

- local event subscription and emission;
- parsed inbound message receipt;
- named cleanup and plugin lifecycle;
- one `Channel` WebSocket connection;
- JSON send/receive and transport lifecycle events.

Plugins own semantics. The bridge itself does not know about `hx-sync`, controls, HTMX, dialog messages, or persistence.

```text
bridge.recv(el, msg, event)  one ingress point for parsed inbound message → local `'msg'` event
bridge.send(msg)             one egress point for semantic message → Channel JSON frame
bridge.addPlugin(name, init) replaceable browser behavior with deterministic cleanup
```

`channel:open`, `channel:close`, and `channel:error` are transport events. They are intentionally separate from control or sync events.

In [ ]:
await add_js_msg('_bridge_js', i_collapsed=True);

In [ ]:
_bridge_js = r'''const bridge = globalThis.bridge || {}

bridge.msgs ||= []
bridge.cbs ||= {}
bridge.cleanups ||= {}

bridge.on = (nm, f) => {
  ;(bridge.cbs[nm] ||= []).push(f)
  return () => bridge.off(nm, f)
}

bridge.off = (nm, f) => {
  if (!f) return delete bridge.cbs[nm]
  bridge.cbs[nm] = (bridge.cbs[nm] || []).filter(g => g!==f)
  if (!bridge.cbs[nm].length) delete bridge.cbs[nm]
}

bridge.emit = (nm, data) => (bridge.cbs[nm] || []).map(f => f(data))

bridge.recv = (el, msg, event) => {
  bridge.msgs.push(msg)
  bridge.emit('msg', {el, msg, event})
  return msg
}

bridge.cleanup = nm => {
  const f = bridge.cleanups[nm]
  delete bridge.cleanups[nm]
  return f?.()
}

bridge.addCleanup = (nm, f) => {
  bridge.cleanup(nm)
  bridge.cleanups[nm] = f
  return () => bridge.cleanup(nm)
}

bridge.addPlugin = (nm, init) => bridge.addCleanup(`plugin:${nm}`, init(bridge) || (() => {}))

bridge.chanUrl = chan => {
  const url = new URL('/wsx', location.href)
  url.searchParams.set('chan', chan)
  url.protocol = location.protocol=='https:' ? 'wss:' : 'ws:'
  return url.href
}

bridge.connect = chan => {
  bridge.cleanup('channel')
  const ws = bridge.ws = new WebSocket(bridge.chanUrl(chan))

  ws.onopen = event => bridge.emit('channel:open', {chan, ws, event})

  ws.onmessage = event => {
    try { bridge.recv(null, JSON.parse(event.data), event) }
    catch (err) { bridge.emit('channel:error', {chan, ws, event, err}) }
  }

  ws.onerror = event => bridge.emit('channel:error', {chan, ws, event})

  ws.onclose = event => {
    if (bridge.ws===ws) bridge.ws = null
    bridge.emit('channel:close', {chan, ws, event})
  }

  bridge.addCleanup('channel', () => {
    if (bridge.ws===ws) bridge.ws = null
    ws.close()
  })

  return ws
}

bridge.send = msg => {
  const ws = bridge.ws
  if (!ws || ws.readyState!=WebSocket.OPEN) return false
  ws.send(JSON.stringify(msg))
  return true
}

globalThis.bridge = bridge
export {bridge}'''
#| linkedto: fc9ce8f3

In [ ]:
await setup_mod('bridge', _bridge_js, True)

test_is(await hx.eval_a('window.bridge && typeof bridge.send==="function"'), True)

Test the generic replacement/cleanup contract before adding any plugin code:

In [ ]:
r = await js_eval_a("""
const seen = []
const rm = bridge.addPlugin('test', () => () => seen.push('first'))
bridge.addPlugin('test', () => () => seen.push('second'))
bridge.cleanup('plugin:test')
rm()
return {seen, cleanups:Object.keys(bridge.cleanups)}
""")

test_eq(r.result, dict(seen=['first', 'second'], cleanups=[]))

Replacing a plugin tears down its predecessor, and a stale cleanup callback cannot tear down the newer plugin.

## declarative hx-sync plugin v0
Defines declarative, bidirectional bindings between supported browser form controls and live kernel state.

`hx-sync` declares a live binding between a supported element control and a dotted kernel path:

```html
<input type="range" hx-sync="state.count">
<select hx-sync="state.choice">
<textarea hx-sync="state.note"></textarea>
```

Browser-originated changes send:

```python
dict(key='sync', op='set', path='state.count', value=17, raw='17')
```

Kernel-originated changes use:

```python
await set_sync('state.count', 17)
```

The sync plugin owns both directions:

```text
sync.send(el, event)  DOM value → normalized sync frame → bridge/Channel
sync.apply(msg)       incoming sync frame → matching DOM properties
```

Initial supported values:

| Element | Normalized value |
|---|---|
| `Input(type='range')` | number |
| `Input(type='number')` | number, or `null` when empty/invalid |
| `Input(type='checkbox')` | boolean |
| other `Input`s | string |
| `Select` | string, or an array for `multiple` |
| `Textarea` | string |

Applying an incoming frame updates DOM properties without dispatching `input` or `change`; this avoids a browser→kernel echo.

### sync plugin
Provides bidirectional live synchronization between declaratively bound browser controls and kernel state.

It owns all `hx-sync` semantics: value conversion, browser-originated messages, incoming application, listeners, and its bridge subscription.

In [ ]:
await add_js_msg('_sync_js', i_collapsed=True);

In [ ]:
_sync_js = r'''bridge.initSync = bridge => {
  const sync = bridge.sync = {}

  sync.val = el => {
    if (el instanceof HTMLInputElement) {
      if (el.type=='checkbox') return el.checked
      if (['range', 'number'].includes(el.type)) {
        const v = el.valueAsNumber
        return Number.isNaN(v) ? null : v
      }
      return el.value
    }
    if (el instanceof HTMLSelectElement) return el.multiple ? [...el.selectedOptions].map(o => o.value) : el.value
    if (el instanceof HTMLTextAreaElement) return el.value
  }

  sync.send = (el, event) => {
    const path = el.getAttribute('hx-sync'), value = sync.val(el)
    if (!path || value===undefined) return
    const msg = {key:'sync', op:'set', path, value, raw:el.value}
    bridge.recv(el, msg, event)
    bridge.send(msg)
    return msg
  }

  sync.apply = msg => {
    if (msg.key!='sync' || msg.op!='set') return
    for (const el of document.querySelectorAll('[hx-sync]')) {
      if (el.getAttribute('hx-sync')!=msg.path) continue
      if (el instanceof HTMLInputElement && el.type=='checkbox') el.checked = msg.value
      else if (el instanceof HTMLSelectElement && el.multiple) {
        for (const o of el.options) o.selected = msg.value.includes(o.value)
      } else if (el instanceof HTMLInputElement && el.type=='number' && msg.value==null) el.value = ''
      else if (el instanceof HTMLInputElement || el instanceof HTMLSelectElement || el instanceof HTMLTextAreaElement) el.value = msg.value
    }
  }

  const change = e => sync.send(e.target, e)
  document.addEventListener('input', change)
  document.addEventListener('change', change)

  const off = bridge.on('msg', ({msg}) => sync.apply(msg))

  return () => {
    document.removeEventListener('input', change)
    document.removeEventListener('change', change)
    off()
    if (bridge.sync===sync) delete bridge.sync
  }
}

bridge.addPlugin('sync', bridge.initSync)'''
#| linkedto: 26b054be

In [ ]:
await setup_mod('sync', _sync_js, True)

test_is(await hx.eval_a('window.bridge.sync && typeof bridge.sync.apply==="function"'), True)

This establishes two intentional plugin APIs:

```javascript
bridge.sync.send(el, event)
bridge.sync.apply(msg)
```

`send` is the shared browser-to-kernel path that a later HTMX extension can call for newly swapped-in `hx-sync` elements. `apply` is the kernel-to-browser path.

The delegated `input` and `change` handlers then become one-line adapters:

```javascript
const change = e => sync.send(e.target, e)
```

That gives the forthcoming HTMX extension one safe, tested entry point: after an HTMX swap, it can call `sync.send(el, event)` for newly introduced `hx-sync` elements without duplicating normalization, local events, or transport handling.

### basic control

The plugin is installed. Before introducing `Channel` and kernel dispatch, verify its browser-local contract with one range input. `bridge.send` will return `false` here because no Channel socket exists yet; `bridge.recv` should still retain the semantic frame.

In [ ]:
Input(id='sync-rng', type='range', min=0, max=25, value=13, hx_sync='state.count')

<input type="range" min="0" max="25" value="13" hx-sync="state.count" id="sync-rng" name="sync-rng">

### browser → kernel test

In [ ]:
r = await js_eval_a("""
bridge.msgs = []
const el = document.getElementById('sync-rng')
el.value = 17
el.dispatchEvent(new Event('input', {bubbles:true}))
return bridge.msgs.at(-1)
""")

test_eq(r.result, dict(key='sync', op='set', path='state.count', value=17, raw='17'))

This establishes that the plugin discovers `hx-sync`, normalizes a range value to a number, and creates the protocol message independently of transport.

### kernel → browser tests

The browser-originated range contract passed without a transport connection.

Now verify the reverse local contract: applying an incoming sync frame updates the element without creating an echo message.

In [ ]:
r = await js_eval_a("""
bridge.msgs = []
const el = document.getElementById('sync-rng')
bridge.sync.apply({key:'sync', op:'set', path:'state.count', value:8})
return {value:el.value, msgs:bridge.msgs}
""")

test_eq(r.result, dict(value='8', msgs=[]))

This completes the plugin’s browser-only contract: `sync.send` produces normalized frames, while `sync.apply` incoming frames changes DOM state silently.

### plugin replacement

Reinstalling the sync plugin must remove its prior delegated listeners. The following browser-local test dispatches one input after replacement and expects exactly one semantic frame.

In [ ]:
r = await js_eval_a("""
bridge.msgs = []
bridge.addPlugin('sync', bridge.initSync)
const el = document.getElementById('sync-rng')
el.value = 11
el.dispatchEvent(new Event('input', {bubbles:true}))
return bridge.msgs.filter(msg => msg.path=='state.count').length
""")

test_eq(r.result, 1)

## kernel bridge
The kernel bridge keeps browser controls and live kernel state synchronized.

`Channel` is a duplex JSON pipe between this kernel and the browser. It owns transport only: not object identity, reconnection policy, subscriptions, persistence, or control semantics.

Exactly one task consumes `ch.events`. It dispatches each frame by `key`, preventing competing consumers from silently taking one another’s messages.

The initial sync handler supports dotted paths through already-defined kernel objects:

```text
state.count
a.b.c
```

This is intentionally narrow and trusted-local-browser only. Bracket paths, lists, arbitrary properties, validation, authorization, and multi-client policy are deferred until their required semantics are clear.

In [ ]:
chan = globals().get('chan') or f'bridge-{simple_id()}'
ch = globals().get('ch')
bridge_task = globals().get('bridge_task')
async def get_ch():
    global ch
    if ch is None: ch = await Channel.connect(chan)
    return ch

handlers = {}

In [ ]:
def recv_sync(msg):
    if msg.get('op')!='set': return
    root,*ks = msg.path.split('.')
    # o = globals()[root]  # this only consider this module not `user_ns`
    o = find_var(root)
    for k in ks[:-1]: o = getattr(o, k)
    setattr(o, ks[-1], msg.value)
    return msg.value

handlers['sync'] = recv_sync

def recv_msg(msg):
    if f := handlers.get(msg.get('key')): return f(msg)

async def watch_bridge(ch):
    while True: recv_msg(await ch.events.get())

In [ ]:
async def start_bridge(ch=None):
    global bridge_task
    if bridge_task and not bridge_task.done():
        bridge_task.cancel()
        try: await bridge_task
        except asyncio.CancelledError: pass
    bridge_task = bg_task(watch_bridge(ch or (await get_ch())))
    return bridge_task

In [ ]:
async def set_sync(path, value, ch=None):
    msg = AD(key='sync', op='set', path=path, value=value)
    recv_sync(msg)
    await (ch or (await get_ch())).send(**msg)
    return value

In [ ]:
ch = await get_ch()

This is the complete kernel-side bridge contract:

- `ch` is the JSON transport endpoint;
- `watch_bridge` is its sole consumer;
- `handlers` routes semantic message keys;
- `recv_sync` updates a dotted kernel path;
- `set_sync` performs a kernel update and broadcasts the same semantic frame.

The `state = AD(...)` setup and all rendered controls remain test fixtures below this cell.


## plugin executable contract tests

These tests protect the bridge’s behavioral boundary rather than its implementation details:

- Channel delivers messages in both directions;
- browser changes update declared kernel state;
- `set_sync` updates declared kernel state and matching DOM elements;
- range, select, checkbox, textarea, and text input share the same contract.

### Channel transport
Explicit Channel setup and a two-way contract test

In [ ]:
await js_eval_a(f"""bridge.connect('{chan}')""")
await wait_for_a(lambda: hx.eval("bridge.ws?.readyState")==1)

test_eq(hx.eval("bridge.ws?.readyState"), 1)

`1` is the browser WebSocket `OPEN` state. This establishes the live connection before adding the kernel dispatcher.

The Channel connection is open. Verify the plugin now forwards a browser-originated sync frame to Python:

In [ ]:
await js_eval_a("""
const el = document.getElementById('sync-rng')
el.value = 18
el.dispatchEvent(new Event('input', {bubbles:true}))
""")

msg = await ch.events.get()
test_eq(dict(msg), dict(key='sync', op='set', path='state.count', value=18, raw='18'))

Browser → Python transport passed: the `hx-sync` frame arrived on `ch.events`.

### kernel dispatch

Single kernel dispatcher and a small declared state object. Start `state.count` at `13` so the next event can prove the assignment changed it.

In [ ]:
state = AD(count=13)
old_task = bridge_task
bridge_task = await start_bridge(ch)
test_is(bridge_task.done(), False)
if old_task: test_is(old_task.done(), True)

State is initialized and the sole Channel consumer is running.

The dispatcher is running. Verify the complete browser → kernel path now updates the declared state:

In [ ]:
await js_eval_a("""
const el = document.getElementById('sync-rng')
el.value = 19
el.dispatchEvent(new Event('input', {bubbles:true}))
""")

await wait_for_a(lambda: state.count==19)
test_eq(await hx.eval_a('$("#sync-rng")[0].value'), '19')
test_eq(state.count, 19)


The browser → kernel route is complete. Verify the full kernel → browser path, which updates kernel state and broadcasts the same sync frame:



In [ ]:
await set_sync('state.count', 12)

await wait_for_a(lambda: hx.eval('$("#sync-rng")[0].value')=='12')
test_eq((state.count, await hx.eval_a('$("#sync-rng")[0].value')), (12, '12'))

This establishes the bidirectional contract for the compact bridge: `hx-sync="state.count"` keeps the live DOM and declared kernel value aligned in either direction.

The compact core now passes its first bidirectional contract:

```text
browser `input`
→ sync plugin
→ Channel
→ dispatcher
→ state.count

set_sync('state.count', value)
→ state.count
→ Channel
→ sync plugin
→ DOM value
```

Before treating the sync plugin as complete, add one compact fixture for its remaining supported form types:

### other types

This keeps the remaining test surface in one rendered fixture: single and multi-select, number, checkbox, textarea, and text input.

In [ ]:
state.choice,state.enabled,state.note,state.title,state.number,state.choices = 'green',False,'hello','draft',7,['red','blue']

Div(
    Select(
        Option('Red', value='red'),
        Option('Green', value='green', selected=True),
        Option('Blue', value='blue'),
        id='sync-select', hx_sync='state.choice',
    ),
    Select(
        Option('Red', value='red', selected=True),
        Option('Green', value='green'),
        Option('Blue', value='blue', selected=True),
        id='sync-selects', multiple=True, hx_sync='state.choices',
    ),
    Input(id='sync-number', type='number', value=state.number, hx_sync='state.number'),
    Input(id='sync-check', type='checkbox', checked=state.enabled, hx_sync='state.enabled'),
    Textarea(state.note, id='sync-note', hx_sync='state.note'),
    Input(id='sync-title', value=state.title, hx_sync='state.title'),
)

<div><select hx-sync="state.choice" id="sync-select" name="sync-select"><option value="red">Red</option><option value="green" selected>Green</option><option value="blue">Blue</option></select><select multiple hx-sync="state.choices" id="sync-selects" name="sync-selects"><option value="red" selected>Red</option><option value="green">Green</option><option value="blue" selected>Blue</option></select><input type="number" value="7" hx-sync="state.number" id="sync-number" name="sync-number"><input type="checkbox" hx-sync="state.enabled" id="sync-check" name="sync-check"><textarea hx-sync="state.note" id="sync-note" name="sync-note">hello</textarea><input value="draft" hx-sync="state.title" id="sync-title" name="sync-title"></div>

In [ ]:
await js_eval_a("""
const sel = $('#sync-select')[0], sels = $('#sync-selects')[0], num = $('#sync-number')[0], chk = $('#sync-check')[0], note = $('#sync-note')[0], title = $('#sync-title')[0]

sel.value = 'blue'
for (const o of sels.options) o.selected = ['green', 'blue'].includes(o.value)
num.value = 9
chk.checked = true
note.value = 'from browser'
title.value = 'browser title'

sel.dispatchEvent(new Event('change', {bubbles:true}))
sels.dispatchEvent(new Event('change', {bubbles:true}))
num.dispatchEvent(new Event('input', {bubbles:true}))
chk.dispatchEvent(new Event('change', {bubbles:true}))
note.dispatchEvent(new Event('input', {bubbles:true}))
title.dispatchEvent(new Event('input', {bubbles:true}))
""")

await wait_for_a(lambda: (state.choice,state.choices,state.number,state.enabled,state.note,state.title)==('blue',['green','blue'],9,True,'from browser','browser title'))
test_eq((state.choice,state.choices,state.number,state.enabled,state.note,state.title), ('blue',['green','blue'],9,True,'from browser','browser title'))

await js_eval_a("""
const num = $('#sync-number')[0]
num.value = ''
num.dispatchEvent(new Event('input', {bubbles:true}))
""")

await wait_for_a(lambda: state.number is None)
test_is(state.number, None)

Browser → kernel coverage now passes for all initially supported control types.

Test the reverse direction:

In [ ]:
await set_sync('state.choice', 'red')
await set_sync('state.choices', ['red','green'])
await set_sync('state.number', 4)
await set_sync('state.enabled', False)
await set_sync('state.note', 'from kernel')
await set_sync('state.title', 'kernel title')

await wait_for_a(lambda: hx.eval("""[
  $('#sync-select')[0].value,
  [...$('#sync-selects')[0].selectedOptions].map(o => o.value),
  $('#sync-number')[0].value,
  $('#sync-check')[0].checked,
  $('#sync-note')[0].value,
  $('#sync-title')[0].value,
]""")==['red',['red','green'],'4',False,'from kernel','kernel title'])

test_eq(
    (state.choice,state.choices,state.number,state.enabled,state.note,state.title),
    ('red',['red','green'],4,False,'from kernel','kernel title'),
)


await set_sync('state.number', None)
await wait_for_a(lambda: hx.eval("$('#sync-number')[0].value")== '')
test_eq((state.number, await hx.eval_a("$('#sync-number')[0].value")), (None, ''))

The sync plugin now has compact bidirectional coverage for:

- range and number values;
- single- and multi-select values;
- checkbox booleans;
- textarea and text strings;
- silent browser application, with no input/change echo.

## Deferred boundaries

- HTMX extension: synchronize `hx-sync` values introduced by `htmx:afterSwap`
- durable dialog projection: source/output persistence
- richer path grammar & control types, validation, lists, radio groups, multi-select
- `idiomorph` and attribute-level updates
- reconnection and multi-client semantics

## next steps

The next design work (in this or separate dialogs):

1. **Refactor bridge**
  Possible `bridge` refactor.

2. **HTMX lifecycle adapter**
  Build a thin HTMX extension that forwards lifecycle events to bridge plugins. The first use is likely `htmx:afterSwap`: identify newly introduced `hx-sync` elements and invoke the sync plugin’s established browser-originated path.

3. **Durable dialog projection**
  Design how a semantic control change becomes durable, model-readable dialog context. Possible targets include `mdhtml` notes, prompt content, code source, code output, raw messages, or structured state companions.

  The key requirement is:

  ```text
  interaction
  → semantic state change
  → durable dialog artifact
  → same state visible after save, reload, kernel restart, and in later LLM context
  ```

4. **Control model and composition**
  Define controls as composable dialog-native units with stable identity, semantic value, rendering, interactions, and durable projection. This work should cover simple choices and buttons first, then forms, trees, tables, graphs, and cross-message context-editing controls.

**Deferred technical questions**
- HTMX swap and mutation policy;
- persistence/hydration of control state;
- richer path grammar and validation;
- radio groups and multi-select semantics;
- reconnecting, peer identity, and multi-client behavior;
- kernel-to-browser DOM patch strategies such as attribute updates or idiomorph.

`bridge` should remain narrow while those layers evolve above it.

## HTMX lifecycle adapter v0

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The most useful next step is the **HTMX lifecycle adapter**. It exercises the existing `bridge.sync.send(el, event)` API without expanding the bridge’s scope.

Its first, narrow contract should be:

```text
HTMX swaps in an element with hx-sync="state.path"
→ adapter finds that element and any matching descendants
→ adapter calls bridge.sync.send(...) once per control
→ existing Channel/dispatcher path updates kernel state
```

This should be a separate plugin: it knows about HTMX events, while `bridge` remains transport/lifecycle infrastructure and `sync` remains control semantics.

The first test should swap in one range input with `hx-sync="state.count"` and verify `state.count` receives its rendered initial value exactly once.

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 50466, "completion_tokens": 675, "total_tokens": 51141, "cached_tokens": 8576, "reasoning_tokens": 516, "cost": 0.00807615}
```


.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Add the narrow adapter as a separate browser plugin. It handles `htmx:afterSwap`, finds the swapped root plus any `hx-sync` descendants, and delegates each to the existing sync path.

```javascript
bridge.initHtmx = bridge => {
  const afterSwap = e => {
    const root = e.detail.elt
    const els = [root, ...root.querySelectorAll?.('[hx-sync]') || []].filter(el => el?.matches?.('[hx-sync]'))
    for (const el of els) bridge.sync?.send(el, e)
  }

  document.body.addEventListener('htmx:afterSwap', afterSwap)
  return () => document.body.removeEventListener('htmx:afterSwap', afterSwap)
}

bridge.addPlugin('htmx', bridge.initHtmx)
```

This does not add new sync semantics or another transport route: every initial value follows the established `sync.send → bridge → Channel → dispatcher` route.

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 51330, "completion_tokens": 515, "total_tokens": 51845, "cached_tokens": 3456, "reasoning_tokens": 308, "cost": 0.008085750000000001}
```


In [ ]:
await add_js_msg('_htmx_js', i_collapsed=True);

In [ ]:
_htmx_js = r'''bridge.initHtmx = bridge => {
  const afterSwap = e => {
    const root = e.detail.elt
    const els = [root, ...root.querySelectorAll?.('[hx-sync]') || []].filter(el => el?.matches?.('[hx-sync]'))
    for (const el of els) bridge.sync?.send(el, e)
  }

  document.body.addEventListener('htmx:afterSwap', afterSwap)
  return () => document.body.removeEventListener('htmx:afterSwap', afterSwap)
}

bridge.addPlugin('htmx', bridge.initHtmx)'''
#| linkedto: 68b0fbfd

In [ ]:
await setup_mod('htmx', _htmx_js, True)

test_is(await hx.eval_a('!!window.bridge.initHtmx'), True)

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Test the adapter’s first contract: a swapped-in `hx-sync` range sends its initial value once through the existing browser → Channel → dispatcher path.

```python
state.count = 13

r = await js_eval_a("""
bridge.msgs = []
const root = document.createElement('div')
root.innerHTML = '<input type="range" value="21" hx-sync="state.count">'
document.body.append(root)
document.body.dispatchEvent(new CustomEvent('htmx:afterSwap', {detail:{elt:root}}))
root.remove()
return bridge.msgs
""")

await wait_for_a(lambda: state.count==21)
test_eq(r.result, [dict(key='sync', op='set', path='state.count', value=21, raw='21')])
test_eq(state.count, 21)
```

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 50881, "completion_tokens": 688, "total_tokens": 51569, "cached_tokens": 3456, "reasoning_tokens": 516, "cost": 0.00814815}
```


In [ ]:
state.count = 13

r = await js_eval_a("""
bridge.msgs = []
const root = document.createElement('div')
root.innerHTML = '<input type="range" value="21" hx-sync="state.count">'
document.body.append(root)
document.body.dispatchEvent(new CustomEvent('htmx:afterSwap', {detail:{elt:root}}))
root.remove()
return bridge.msgs
""")

await wait_for_a(lambda: state.count==21)
test_eq(r.result, [dict(key='sync', op='set', path='state.count', value=21, raw='21')])
test_eq(state.count, 21)

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The first test covers an `hx-sync` descendant within a swapped container. Test the other declared case: the swapped root is itself the control.

```python
state.count = 13

r = await js_eval_a("""
bridge.msgs = []
const root = document.createElement('input')
root.type = 'range'
root.value = 22
root.setAttribute('hx-sync', 'state.count')
document.body.append(root)
document.body.dispatchEvent(new CustomEvent('htmx:afterSwap', {detail:{elt:root}}))
root.remove()
return bridge.msgs
""")

await wait_for_a(lambda: state.count==22)
test_eq(r.result, [dict(key='sync', op='set', path='state.count', value=22, raw='22')])
test_eq(state.count, 22)
```

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 51252, "completion_tokens": 495, "total_tokens": 51747, "cached_tokens": 50560, "reasoning_tokens": 321, "cost": 0.00805905}
```


In [ ]:
Div(id='bridge-arena01')#(Input(type='range', min=0, max=25, value=13, hx_sync='state.count'))

<div id="bridge-arena01"></div>

In [ ]:
# state.count = 13
# 
# r = await js_eval_a("""
# bridge.msgs = []
# const el = $('#bridge-arena01 input')[0]
# document.body.dispatchEvent(new CustomEvent('htmx:afterSwap', {detail:{elt:el}}))
# root.remove()
# return bridge.msgs
# """)
# 
# await wait_for_a(lambda: state.count==22)
# test_eq(r.result, [dict(key='sync', op='set', path='state.count', value=22, raw='22')])
# test_eq(state.count, 22)

In [ ]:
state.count = 13

r = await js_eval_a("""
bridge.msgs = []
const arena = document.getElementById('bridge-arena01')
const root = document.createElement('input')
root.type = 'range'
root.value = 22
root.setAttribute('hx-sync', 'state.count')
arena.append(root)
arena.dispatchEvent(new CustomEvent('htmx:afterSwap', {bubbles:true, detail:{elt:root}}))
root.remove()
return bridge.msgs
""")

await wait_for_a(lambda: state.count==22)
test_eq(r.result, [dict(key='sync', op='set', path='state.count', value=22, raw='22')])
test_eq(state.count, 22)

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Both declared discovery cases now pass: the adapter handles a swapped control itself and controls nested inside a swapped container.

The remaining compact contract is that it discovers **every** matching descendant, not only the first:

```python
state.count,state.title = 13,'draft'

r = await js_eval_a(r"""
bridge.msgs = []
const arena = document.getElementById('bridge-arena01')
const root = document.createElement('div')
root.innerHTML = `
  <input type="range" value="23" hx-sync="state.count">
  <input value="from swap" hx-sync="state.title">
`
arena.append(root)
arena.dispatchEvent(new CustomEvent('htmx:afterSwap', {bubbles:true, detail:{elt:root}}))
root.remove()
return bridge.msgs
""")

await wait_for_a(lambda: (state.count,state.title)==(23,'from swap'))
test_eq(r.result, [
    dict(key='sync', op='set', path='state.count', value=23, raw='23'),
    dict(key='sync', op='set', path='state.title', value='from swap', raw='from swap'),
])
test_eq((state.count,state.title), (23,'from swap'))
```

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 51728, "completion_tokens": 776, "total_tokens": 52504, "cached_tokens": 8576, "reasoning_tokens": 516, "cost": 0.008341200000000002}
```


In [ ]:
state.count,state.title = 13,'draft'

r = await js_eval_a(r"""
bridge.msgs = []
const arena = document.getElementById('bridge-arena01')
const root = document.createElement('div')
root.innerHTML = `
  <input type="range" value="23" hx-sync="state.count">
  <input value="from swap" hx-sync="state.title">
`
arena.append(root)
arena.dispatchEvent(new CustomEvent('htmx:afterSwap', {bubbles:true, detail:{elt:root}}))
root.remove()
return bridge.msgs
""")

await wait_for_a(lambda: (state.count,state.title)==(23,'from swap'))
test_eq(r.result, [
    dict(key='sync', op='set', path='state.count', value=23, raw='23'),
    dict(key='sync', op='set', path='state.title', value='from swap', raw='from swap'),
])
test_eq((state.count,state.title), (23,'from swap'))

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The adapter now synchronizes every `hx-sync` descendant. Test plugin replacement next: reinstalling the HTMX adapter must not leave two `htmx:afterSwap` listeners behind.

```python
state.count = 13
await js_eval_a(r"""bridge.addPlugin('htmx', bridge.initHtmx)""")

r = await js_eval_a(r"""
bridge.msgs = []
const arena = document.getElementById('bridge-arena01')
const root = document.createElement('input')
root.type = 'range'
root.value = 24
root.setAttribute('hx-sync', 'state.count')
arena.append(root)
arena.dispatchEvent(new CustomEvent('htmx:afterSwap', {bubbles:true, detail:{elt:root}}))
root.remove()
return bridge.msgs.filter(o => o.path=='state.count')
""")

await wait_for_a(lambda: state.count==24)
test_eq(r.result, [dict(key='sync', op='set', path='state.count', value=24, raw='24')])
```

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 52260, "completion_tokens": 434, "total_tokens": 52694, "cached_tokens": 51584, "reasoning_tokens": 214, "cost": 0.008164500000000002}
```


In [ ]:
state.count = 13
await js_eval_a(r"""bridge.addPlugin('htmx', bridge.initHtmx)""")

r = await js_eval_a(r"""
bridge.msgs = []
const arena = document.getElementById('bridge-arena01')
const root = document.createElement('input')
root.type = 'range'
root.value = 24
root.setAttribute('hx-sync', 'state.count')
arena.append(root)
arena.dispatchEvent(new CustomEvent('htmx:afterSwap', {bubbles:true, detail:{elt:root}}))
root.remove()
return bridge.msgs.filter(o => o.path=='state.count')
""")

await wait_for_a(lambda: state.count==24)
test_eq(r.result, [dict(key='sync', op='set', path='state.count', value=24, raw='24')])

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The HTMX adapter contract is complete:

- detects a swapped root that is itself an `hx-sync` control;
- detects all `hx-sync` descendants of a swapped container;
- forwards each through the established `sync.send → Channel → dispatcher` route;
- survives plugin replacement without duplicate `htmx:afterSwap` listeners.

It remains appropriately narrow: no mutation observer, persistence, reconnect handling, or new sync semantics.

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 52718, "completion_tokens": 589, "total_tokens": 53307, "cached_tokens": 51584, "reasoning_tokens": 497, "cost": 0.008349450000000001}
```


Are we done here? If so, give me the closing note and then we'll go to the next task.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

```markdown
## HTMX lifecycle adapter complete

`bridge` now has a narrow HTMX adapter that synchronizes `hx-sync` controls introduced by `htmx:afterSwap`.

It:

- handles a swapped root that is itself an `hx-sync` control;
- discovers every `hx-sync` descendant of a swapped container;
- routes each initial value through the existing `sync.send → Channel → dispatcher` path;
- cleans up correctly when the plugin is replaced.

The adapter adds no persistence, mutation-observer policy, reconnection behavior, or new control semantics. Those remain separate later layers.
```

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 52885, "completion_tokens": 223, "total_tokens": 53108, "cached_tokens": 52608, "reasoning_tokens": 95, "cost": 0.0081}
```


**HTMX lifecycle adapter complete**

`bridge` now has a narrow HTMX adapter that synchronizes `hx-sync` controls introduced by `htmx:afterSwap`.

It:

- handles a swapped root that is itself an `hx-sync` control;
- discovers every `hx-sync` descendant of a swapped container;
- routes each initial value through the existing `sync.send → Channel → dispatcher` path;
- cleans up correctly when the plugin is replaced.

The adapter adds no persistence, mutation-observer policy, reconnection behavior, or new control semantics. Those remain separate later layers.

## Durable dialog projection v0

Start with a deliberately narrow projection contract: **one synced path updates one existing note through a declared renderer**.

```text
state.target changes
→ projection registered for state.target
→ renderer produces durable Markdown
→ that note is updated
```

The first version should exclude code-source edits, output edits, message creation, multiple targets, and generic object serialization. A minimal declaration could look like:

```python
project_sync('state.target', note_id, lambda v: f'''Selected deployment target: {v}''')
```

The resulting flow becomes:

```text
browser control
→ sync frame
→ state.target assignment
→ durable note update
→ existing Channel frame updates matching DOM controls
```

`set_sync('state.target', value)` should use the same projection path, so browser- and kernel-originated changes produce identical durable state. The next step is to define the small registration record and decide exactly when the async note update runs.

Use ordered async dispatch rather than a background task. A background projection could race later updates; awaiting it keeps kernel state, durable note, and subsequent frames in one sequence. The browser control still changes immediately, but `set_sync(...)` should not return until its declared projection succeeds.

Redefine `project_sync`, now it only updates but not create projections. `project_sync` should remain the registration API.

In [ ]:
sync_projs = {}
def project_sync(path, msg_id, render): sync_projs[path] = AD(msg_id=msg_id, render=render)

The update operation needs a different name:

In [ ]:
async def apply_sync_proj(path, value):
    if proj := sync_projs.get(path): await update_msg(proj.msg_id, content=proj.render(value))

Redefine the dispatch segment as an ordered async chain.

The next cell can make `recv_sync`, `recv_msg`, and `watch_bridge` async, then await a projection after assigning the kernel value.

In [ ]:
async def recv_sync(msg):
    if msg.get('op')!='set': return
    root,*ks = msg.path.split('.')
    o = find_var(root)
    for k in ks[:-1]: o = getattr(o, k)
    setattr(o, ks[-1], msg.value)
    await apply_sync_proj(msg.path, msg.value)
    return msg.value

handlers['sync'] = recv_sync

async def recv_msg(msg):
    if f := handlers.get(msg.get('key')): return await maybe_await(f(msg))

async def watch_bridge(ch):
    while True: await recv_msg(await ch.events.get())

async def set_sync(path, value, ch=None):
    msg = AD(key='sync', op='set', path=path, value=value)
    await recv_sync(msg)
    await (ch or (await get_ch())).send(**msg)
    return value

Then the revised receiver calls `apply_sync_proj` after updating live kernel state. That preserves the intended split:

```python
project_sync(path, msg_id, render)
```

registers a projection, while:

```python
await apply_sync_proj(path, value)
```

renders and writes the registered durable message.

Restarting `bridge_task` matters: the existing task retains the old `watch_bridge` coroutine.

In [ ]:
bridge_task = await start_bridge(ch)
test_is(bridge_task.done(), False)

Traceback (most recent call last):


  File "/tmp/ipymini_246/1850587646.py", line 16, in watch_bridge
    while True: await recv_msg(await ch.events.get())
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


  File "/tmp/ipymini_246/1850587646.py", line 13, in recv_msg
    if f := handlers.get(msg.get('key')): return await maybe_await(f(msg))
                                                 ^^^^^^^^^^^^^^^^^^^^^^^^^


  File "/usr/local/lib/python3.12/site-packages/fastcore/aio.py", line 88, in maybe_await
    return await o if isawaitable(o) else o
           ^^^^^^^


  File "/tmp/ipymini_246/1850587646.py", line 6, in recv_sync
    setattr(o, ks[-1], msg.value)
               ~~^^^^


IndexError: list index out of range


No projection occurs yet because `sync_projs` remains empty. The revised path now awaits a registered note update before forwarding a kernel-originated change to the browser.

The ordered dispatch is installed. The next test needs a dedicated durable target:

Projected count: 13

Projected count: 15

In [ ]:
proj_id = (await read_msg()).id
project_sync('state.count', proj_id, lambda v: f'''Projected count: {v}''')
test_eq(sync_projs['state.count'].msg_id, proj_id)

Using a dedicated fixture keeps the projection test from modifying an architectural note, and makes the intended durable change explicit before the first write.

The projection is registered against the dedicated note #_6e9219be. Test the kernel-originated route first: `set_sync` should update live state, await the durable note update, then broadcast the sync frame.

In [ ]:
await set_sync('state.count', 14)

proj = await read_msg(0,id=proj_id)
test_eq((state.count, proj['content']), (14, 'Projected count: 14'))

Test the browser-originated route. The dispatcher should await the note update before completing its handling of the incoming frame.

In [ ]:
prev_count = state.count

await js_eval_a("""
const el = document.getElementById('sync-rng')
el.value = 15
el.dispatchEvent(new Event('input', {bubbles:true}))
""")

for _ in range(20):
    proj = await read_msg(0, id=proj_id)
    if proj['content']=='Projected count: 15': break
    await asyncio.sleep(.05)
# await wait_for_a(lambda: state.count!=prev_count)

test_eq((state.count, proj['content']), (15, 'Projected count: 15'))

The two routes now update the same durable target. Add one boundary test: an unregistered path must update live state without touching the registered projection.

In [ ]:
await set_sync('state.title', 'unprojected')

proj = await read_msg(0, id=proj_id)
test_eq((state.title, proj['content']), ('unprojected', 'Projected count: 15'))

This pins down that projection is opt-in by exact path: `state.count` updates #_6e9219be, while `state.title` does not.

The v0 durable-projection contract now passes:

- `project_sync(path, msg_id, render)` registers an exact-path projection;
- browser-originated and `set_sync(...)` changes both update live kernel state and the same durable note;
- the note update is awaited before dispatch completes;
- unregistered paths remain live-only;
- the fixture note is now the durable record for `state.count`: #_6e9219be.

The remaining boundaries are intentional: one projection per path, one existing target message, caller-supplied renderer, and no projection error policy or multi-target composition yet.

###### start-projection options discussion

In [ ]:
get_ipython().user_ns is globals()

True

A little detour (we're not going to implement it in this session): what options do we have to implement `state.count = 17` so the assignment (everywhere) triggers `set_sync()`? ipython hooks? overriding `user_ns`? descriptors?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

`state.count = 17` is an ordinary synchronous attribute assignment, whereas `set_sync()` is async and has side effects: durable-message writes and Channel broadcast. That mismatch is the central constraint.

The practical options are:

1. **Observable state object — best for a deliberate state model**

   Make `state` an object whose `__setattr__` or descriptor-backed fields observe assignments.

   ```python
   class SyncState:
       def __setattr__(self, k, v):
           object.__setattr__(self, k, v)
           bg_task(set_sync(f'state.{k}', v))
   ```

   Or define individual descriptor/property fields for stronger validation and explicit per-field policy.

   This catches `state.count = 17` wherever it occurs, including normal Python functions, provided all writes go through that state object. It does **not** catch arbitrary nested mutation such as `state.filters.append('x')` unless those values are observable too.

   The complication is async ordering and errors: assignment returns immediately, so a background `set_sync` can race with later assignments and failures are detached from the caller. A robust design would separate:
   - a raw local assignment used by `recv_sync`;
   - an observable assignment that schedules or queues propagation;
   - loop suppression when a browser-originated frame is being applied.

2. **Explicit async API — best correctness boundary**

   Keep:

   ```python
   await set_sync('state.count', 17)
   ```

   This makes ordering, projection failures, and browser delivery observable to the caller. It is the simplest and most reliable contract, but requires callers to opt in rather than making normal assignment magical.

3. **IPython AST/input transformation — convenient notebook syntax**

   An IPython transformer could rewrite:

   ```python
   state.count = 17
   ```

   into:

   ```python
   await set_sync('state.count', 17)
   ```

   This preserves awaited semantics for interactive cells. But it only covers code submitted through that configured IPython execution path; it will not reliably cover imported modules, dynamically executed code, callbacks, or assignments originating elsewhere. It is best viewed as ergonomic syntax layered over the explicit API, not enforcement.

4. **Custom `user_ns` / globals dictionary — limited to top-level names**

   A custom namespace mapping could observe:

   ```python
   state = new_state
   ```

   but not:

   ```python
   state.count = 17
   ```

   Once Python resolves `state`, the attribute write happens on the object. Since `user_ns is globals()` here, replacing or wrapping it would also be invasive and still would not solve nested attributes. It is not the right mechanism for this problem.

5. **IPython hooks — observation, not assignment interception**

   IPython has pre-/post-cell execution hooks and display/event hooks. They can inspect state before and after a cell, perhaps diff a declared state object, then synchronize detected changes. They cannot directly intercept each `STORE_ATTR` operation. This enables coarse reconciliation, but loses precise ordering and struggles with in-place mutation.

6. **Tracing/profiling or bytecode instrumentation — not suitable**

   Python tracing does not provide a clean, general callback for every attribute-store operation. Bytecode or AST instrumentation could theoretically rewrite assignments more broadly, but it is fragile, incomplete across execution sources, and disproportionate for a dialog-control system.

So the likely architecture is:

```text
durable, ordered changes:       await set_sync(...)
normal ergonomic state writes:  observable SyncState fields
notebook convenience:           optional AST rewrite to await set_sync(...)
```

If “everywhere” means every regular `state.count = value` in code you control, a purpose-built observable `state` object is the only credible general mechanism. If it means every possible mutation in Python, that is not enforceable without constraining the state model and mutation APIs.

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 55930, "completion_tokens": 1181, "total_tokens": 57111, "cached_tokens": 54656, "reasoning_tokens": 364, "cost": 0.009275249999999999}
```


Ok, but then we enter the reign of `traitlets` or other `observers` libraries that I wanted to avoid.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes, that is exactly the trap to avoid. Traitlets is built around declarative fields and observer registration; adopting it would turn a small bridge concern into a general reactive-object framework. ([traitlets.readthedocs.io](https://traitlets.readthedocs.io/en/stable/api.html?utm_source=openai))

The better[*](https://traitlets.readthedocs.io/en/stable/api.html?utm_source=openai "Traitlets API reference — traitlets 5.15.1 documentation") conclusion is: **do not make arbitrary Python assignment reactive.**

Keep the present division:

```python
state.count = 17                 # ordinary local mutation
await set_sync('state.count', 17) # ordered, durable, broadcast mutation
```

If the repeated path string becomes awkward, provide a thin explicit façade rather than an observer system:

```python
async def set_count(v): return await set_sync('state.count', v)
```

Or, for a compact state object API:

```python
await state.set('count', 17)
```

That retains the important properties:

- no hidden background task;
- callers can await projection and transport errors;
- no echo-suppression machinery buried in `__setattr__`;
- ordinary Python objects remain ordinary;
- durable writes occur only where code explicitly requests them.

An IPython input transform remains a plausible *optional convenience* later—rewriting notebook-cell assignment syntax into `await set_sync(...)`—but it should not become the system’s semantic foundation. The explicit `set_sync` operation is the clean boundary.

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 60620, "completion_tokens": 460, "total_tokens": 61080, "cached_tokens": 55680, "reasoning_tokens": 167, "cost": 0.009438000000000002}
```


###### end-projection options discussion

**Durable dialog projection v0 complete**

`bridge` now supports opt-in durable projection for a synced kernel path.

```python
project_sync(path, msg_id, render)
```

registers an exact-path projection to an existing dialog message. On either a browser-originated sync event or `set_sync(...)`, the bridge:

```text
updates live kernel state
→ awaits the rendered message update
→ sends the matching sync frame to the browser
```

Coverage confirms that:

- browser and kernel changes update the same durable note;
- projection is opt-in and path-specific;
- unregistered paths remain live-only.

This remains a narrow primitive: one target message per path, caller-supplied rendering, and no multi-target composition or projection-failure policy.

## Control model and composition exploration

In [ ]:
state.count=13
state

```python
{ 'choice': 'red',
  'choices': ['red', 'green'],
  'count': 13,
  'enabled': False,
  'note': 'from kernel',
  'number': None,
  'title': 'unprojected'}
```

In [ ]:
Div(id='bridge-arena02', hx_sync='state')(
    Input(type='range', min=0, max=25, value=13, hx_sync='count'),
    Input(value='unprojected', hx_sync='title'),
)

<div hx-sync="state" id="bridge-arena02"><input type="range" min="0" max="25" value="13" hx-sync="count"><input value="unprojected" hx-sync="title"></div>

Let's make this a little exploration to see what'd it take. The v0 implementation will surely happen elsewhere.

Assume we want to sync kernel `state` as a whole.

`state` needs a **group binding**, not a scalar binding. A whole-state update should fan out to its child controls:

```text
{path:'state', value:{count:17, title:'revised'}}
→ state.count control becomes 17
→ state.title control becomes "revised"
```

The existing `sync.apply` should first factor its per-control assignment into a helper, then add one branch for a scope-level snapshot. Browser-originated events can remain leaf-level frames; only whole-state kernel updates need snapshot fan-out.

One important kernel boundary: the current `recv_sync` assumes a path has at least one attribute after its root. Therefore `set_sync('state', state)` cannot work unchanged; root-only paths need an explicit assignment/reconciliation rule. For this exploration, treat whole-state frames as kernel-to-browser snapshots first, without routing them through `recv_sync`.

In `HTMX` many attributes are inherited implicitly (that'll change somehow in HTMX v4). We could perfectly interpret `hx_sync` as inherited so that `hx_sync='state'` on the parent means that children inherit the prefix. `hx-sync` can carry both meanings by element type and ancestry:

```python
Div(hx_sync='state')(
    Input(type='range', hx_sync='count'),
    Input(hx_sync='title'),
)
```

Here the `Div` establishes the inherited prefix; the controls provide relative paths:

```text
count → state.count
title → state.title
```

Nested containers can extend it naturally:

```html
<div hx-sync="state">
  <div hx-sync="settings">
    <input hx-sync="theme">
  </div>
</div>
```

```text
theme → state.settings.theme
```

The browser plugin needs a path resolver rather than treating `el.getAttribute('hx-sync')` as the complete path:

```javascript
sync.path = el => [...el.closest('[hx-sync]') ? [] : []]
```

Conceptually, it should walk from the control through ancestor elements carrying `hx-sync`, collect each segment, reverse them, and join with `.`. A supported control contributes its own segment; non-control elements contribute scope segments.

That supports both directions without a new attribute:

```text
browser input on control hx-sync="count"
→ resolve path "state.count"
→ send normal leaf frame

kernel frame path "state.count"
→ resolve each control’s path
→ apply to matching control

kernel snapshot path "state", value={count:17, title:"revised"}
→ find controls beneath the state-bound container
→ apply snapshot.count and snapshot.title
```

The important distinction is that this is **custom hierarchical interpretation**, not raw HTML attribute inheritance. A child’s `hx-sync="count"` overrides the literal attribute value it would otherwise inherit, but the plugin can deliberately combine it with ancestor declarations. That is a better fit for the control-composition model.

I assume `HTMX` cache the inheritance chain in the element cache / data. We could do something similar.

In fact, `HTMX` extension can access the full element cache (and the internal API) easily and safely.
```javascript
(function() {
  /** @type {import("../htmx").HtmxInternalApi} */
  var api

  htmx.defineExtension('ws', {

    /**
     * init is called once, when this extension is first registered.
     * @param {import("../htmx").HtmxInternalApi} apiRef
     */
    init: function(apiRef) {
      // Store reference to internal API
      api = apiRef

      // Default function for creating new EventSource objects
      if (!htmx.createWebSocket) {
        htmx.createWebSocket = createWebSocket
      }
      ...
```

An HTMX extension is a sensible place to compile and cache the resolved `hx-sync` metadata as HTMX processes elements.

The key point is that `hx-sync` is not necessarily an HTMX-inherited attribute by default, but the extension can define its own inheritance rule and store the result in each element’s HTMX-associated data. That avoids walking ancestors on every `input`, `change`, or incoming frame.

Conceptually, each relevant element gets compiled metadata such as:

```javascript
{
  local: 'count',
  path: 'state.count',
  rel: ['count'],
  scope: 'state'
}
```

For:

```html
<div hx-sync="state">
  <input hx-sync="count">
</div>
```

and nested scopes naturally compose:

```html
<div hx-sync="state">
  <div hx-sync="settings">
    <input hx-sync="theme">
  </div>
</div>
```

```text
input.local = "theme"
input.path  = "state.settings.theme"
input.rel   = ["settings", "theme"]
```

That gives both directions a cheap, unambiguous operation:

```text
browser event
→ use cached input.path
→ send {path: "state.settings.theme", value: ...}

kernel leaf update: path "state.settings.theme"
→ compare against cached input.path
→ apply to matching controls

kernel snapshot: path "state", value={...}
→ find controls under the state scope
→ use each control's cached rel path to read its value from the snapshot
→ apply it
```

The cache should be treated as derived DOM metadata, not durable control state. On an HTMX swap, newly processed elements receive fresh metadata; removed elements disappear with their DOM. If an existing element’s `hx-sync` attribute is changed in place, that is outside the current contract unless the adapter deliberately recompiles it.

A good exploration test would assert behavior rather than cache internals:

```text
nested inherited paths resolve correctly;
a leaf browser event sends the full resolved path;
a root `state` snapshot fans out to descendants using their relative paths;
a swapped subtree receives fresh compiled paths.
```

This keeps the inheritance mechanism inside the HTMX adapter while `bridge.sync` continues to operate on fully resolved paths.

First inspect the per-element HTMX data without defining any inheritance behavior. This establishes whether HTMX has already cached relevant ancestry for the existing controls.

In [ ]:
r = await js_eval_a(r"""
const getData = htmx._('getInternalData')
const els = [...document.querySelectorAll('#bridge-arena02, #bridge-arena02 [hx-sync]')]

return els.map(el => {
  const data = getData(el)
  return {tag:el.tagName, sync:el.getAttribute('hx-sync'), keys:Object.keys(data)}
})
""")

r.result

[{'tag': 'DIV', 'sync': 'state', 'keys': []}, {'tag': 'INPUT', 'sync': 'count', 'keys': []}, {'tag': 'INPUT', 'sync': 'title', 'keys': []}]

This uses HTMX’s internal-data accessor only for inspection. The result will show what metadata exists on the wrapper and controls before deciding whether to cache a resolved `hx-sync` path there.

The HTMX internal-data objects are empty. Since these elements have no built-in HTMX behavior, HTMX has not populated a cache for them.

The next small exploration can prove that the same per-element store is usable for our derived path cache. This compiles each control’s full path once, then returns the cached value on later calls:

In [ ]:
r = await js_eval_a(r"""
const getData = htmx._('getInternalData')

const syncPath = el => {
  const data = getData(el)
  if (data.syncPath) return data.syncPath

  const parts = []
  for (let o=el; o; o=o.parentElement) {
    const part = o.getAttribute('hx-sync')
    if (part) parts.push(part)
  }

  return data.syncPath = parts.reverse().join('.')
}

const els = [...document.querySelectorAll('#bridge-arena02 input')]
return els.map(el => ({
  local: el.getAttribute('hx-sync'),
  path: syncPath(el),
  cached: getData(el).syncPath,
}))
""")

test_eq(r.result, [
    dict(local='count', path='state.count', cached='state.count'),
    dict(local='title', path='state.title', cached='state.title'),
])

This establishes a minimal composition rule without an extension: ancestor and leaf segments become a full path, while `syncPath` remains derived, disposable DOM metadata.

The simple inherited prefix works. Next, test nested scopes and confirm each control keeps its own resolved path:

In [ ]:
r = await js_eval_a(r"""
const getData = htmx._('getInternalData')

const syncPath = el => {
  const data = getData(el)
  if (data.syncPath) return data.syncPath

  const parts = []
  for (let o=el; o; o=o.parentElement) {
    const part = o.getAttribute('hx-sync')
    if (part) parts.push(part)
  }

  return data.syncPath = parts.reverse().join('.')
}

const root = document.createElement('div')
root.setAttribute('hx-sync', 'state')
root.innerHTML = `
  <div hx-sync="settings">
    <input hx-sync="theme">
  </div>
  <input hx-sync="title">
`

const [theme,title] = root.querySelectorAll('input')
return [theme,title].map(el => ({
  local: el.getAttribute('hx-sync'),
  path: syncPath(el),
  cached: getData(el).syncPath,
}))
""")

test_eq(r.result, [
    dict(local='theme', path='state.settings.theme', cached='state.settings.theme'),
    dict(local='title', path='state.title', cached='state.title'),
])

This remains browser-local exploration: it establishes the path-compilation rule before changing `sync.send` or `sync.apply`.

The path rule now handles nested scopes. Next, establish the inverse operation needed for a whole-`state` snapshot: derive each control’s path relative to the incoming snapshot root, then read its value.

In [ ]:
r = await js_eval_a(r"""
const getData = htmx._('getInternalData')

const syncPath = el => {
  const data = getData(el)
  if (data.syncPath) return data.syncPath

  const parts = []
  for (let o=el; o; o=o.parentElement) {
    const part = o.getAttribute('hx-sync')
    if (part) parts.push(part)
  }

  return data.syncPath = parts.reverse().join('.')
}

const getAt = (o, ks) => ks.reduce((o, k) => o?.[k], o)
const root = document.createElement('div')
root.setAttribute('hx-sync', 'state')
root.innerHTML = `
  <div hx-sync="settings"><input hx-sync="theme"></div>
  <input hx-sync="title">
`

const snap = {settings:{theme:'dark'}, title:'revised'}
const els = [...root.querySelectorAll('input')]

return els.map(el => {
  const path = syncPath(el), rel = path.slice('state.'.length).split('.')
  return {path, rel, value:getAt(snap, rel)}
})
""")

test_eq(r.result, [
    dict(path='state.settings.theme', rel=['settings','theme'], value='dark'),
    dict(path='state.title', rel=['title'], value='revised'),
])

This defines the kernel-to-browser snapshot operation without changing the plugin: a frame for `state` can fan out by resolving each descendant’s cached full path relative to `state`.

The snapshot lookup works for present values. The next decision is missing-value policy: a partial `state` snapshot should **not** overwrite a control with JavaScript `undefined`.

Explore that boundary before designing `sync.apply` fan-out:

In [ ]:
r = await js_eval_a(r"""
const getAt = (o, ks) => ks.reduce((o, k) => o?.[k], o)
const snap = {settings:{theme:'dark'}}

return [
  getAt(snap, ['settings','theme']),
  getAt(snap, ['title'])===undefined,
]
""")

test_eq(r.result, ['dark', True])

That suggests the safe whole-state rule:

```text
snapshot has descendant value → apply it
snapshot lacks descendant value → leave that control unchanged
```

It avoids turning a partial kernel update into a destructive browser reset.

Next establish which elements a `state` snapshot should target: only supported leaf controls, not scope containers.

In [ ]:
r = await js_eval_a(r"""
const isControl = el =>
  el instanceof HTMLInputElement ||
  el instanceof HTMLSelectElement ||
  el instanceof HTMLTextAreaElement

const root = document.createElement('div')
root.setAttribute('hx-sync', 'state')
root.innerHTML = `
  <div hx-sync="settings">
    <input hx-sync="theme">
  </div>
  <input hx-sync="title">
`

return [...root.querySelectorAll('[hx-sync]')]
  .filter(isControl)
  .map(el => el.getAttribute('hx-sync'))
""")

test_eq(r.result, ['theme', 'title'])

That gives snapshot fan-out a clean target set: scope containers contribute path segments; only supported controls receive values.

The remaining routing boundary is prefix matching. A snapshot for `state` must target `state.title`, but not `stateful.title`.

In [ ]:
r = await js_eval_a(r"""
const under = (path, root) => path.startsWith(`${root}.`)

return [
  under('state.title', 'state'),
  under('state.settings.theme', 'state'),
  under('stateful.title', 'state'),
  under('state', 'state'),
]
""")

test_eq(r.result, [True, True, False, False])

This yields a precise fan-out rule: a whole-object frame at `state` applies only to leaf controls whose cached full path starts with `state.`. The scope container itself is not a target.

The browser-side shape is now clear:

```text
hx-sync scope chain
→ cached full path per leaf control
→ root snapshot selects paths beneath `state.`
→ relative path reads a value from the snapshot
→ missing values leave controls unchanged
```

One compact integration probe can combine those rules without changing `sync.apply`:

In [ ]:
r = await js_eval_a(r"""
const getData = htmx._('getInternalData')
const isControl = el => el instanceof HTMLInputElement || el instanceof HTMLSelectElement || el instanceof HTMLTextAreaElement
const under = (path, root) => path.startsWith(`${root}.`)
const getAt = (o, ks) => ks.reduce((o, k) => o?.[k], o)

const syncPath = el => {
  const data = getData(el)
  if (data.syncPath) return data.syncPath

  const parts = []
  for (let o=el; o; o=o.parentElement) {
    const part = o.getAttribute('hx-sync')
    if (part) parts.push(part)
  }

  return data.syncPath = parts.reverse().join('.')
}

const root = document.createElement('div')
root.setAttribute('hx-sync', 'state')
root.innerHTML = `
  <div hx-sync="settings"><input hx-sync="theme"></div>
  <input hx-sync="title">
`

const snap = {settings:{theme:'dark'}}
return [...root.querySelectorAll('[hx-sync]')].filter(isControl)
  .map(el => {
    const path = syncPath(el), rel = path.slice('state.'.length).split('.'), value = getAt(snap, rel)
    return {path, value, apply:under(path, 'state') && value!==undefined}
  })
""")

test_eq(r.result, [
    dict(path='state.settings.theme', value='dark', apply=True),
    dict(path='state.title', apply=False),
])

This confirms a partial `state` snapshot would update `state.settings.theme` while leaving `state.title` untouched.

The integration probe passes. JavaScript JSON serialization omits an `undefined` property, so the missing `value` in the `state.title` result is expected.

The next boundary is cache invalidation. If a scoped ancestor’s `hx-sync` changes after a child path is cached, the cached child path remains stale unless the adapter explicitly recompiles it.

In [ ]:
r = await js_eval_a(r"""
const getData = htmx._('getInternalData')

const syncPath = el => {
  const data = getData(el)
  if (data.syncPath) return data.syncPath

  const parts = []
  for (let o=el; o; o=o.parentElement) {
    const part = o.getAttribute('hx-sync')
    if (part) parts.push(part)
  }

  return data.syncPath = parts.reverse().join('.')
}

const root = document.createElement('div')
root.setAttribute('hx-sync', 'state')
root.innerHTML = '<input hx-sync="count">'

const el = root.querySelector('input')
const before = syncPath(el)
root.setAttribute('hx-sync', 'other')

return {before, after:syncPath(el), cached:getData(el).syncPath}
""")

test_eq(r.result, dict(before='state.count', after='state.count', cached='state.count'))

That result would support a clean v0 rule: `hx-sync` scope attributes are compiled when the element is introduced, and changing them in place requires explicit recompilation or an HTMX swap.

That establishes the lifecycle boundary: resolved paths are cached derived metadata, so an in-place change to an `hx-sync` scope does not take effect until the cache is explicitly cleared or the subtree is replaced.

The next useful probe is the browser-event frame itself: confirm a leaf control’s cached inherited path replaces its local `hx-sync` segment.

In [ ]:
r = await js_eval_a(r"""
const getData = htmx._('getInternalData')

const syncPath = el => {
  const data = getData(el)
  if (data.syncPath) return data.syncPath

  const parts = []
  for (let o=el; o; o=o.parentElement) {
    const part = o.getAttribute('hx-sync')
    if (part) parts.push(part)
  }

  return data.syncPath = parts.reverse().join('.')
}

const root = document.createElement('div')
root.setAttribute('hx-sync', 'state')
root.innerHTML = '<input type="range" value="17" hx-sync="count">'

const el = root.querySelector('input')
return ({
  local: el.getAttribute('hx-sync'),
  path: syncPath(el),
  value: el.valueAsNumber,
  raw: el.value,
})
""")

test_eq(r.result, dict(local='count', path='state.count', value=17, raw='17'))

This isolates the change a future `sync.send` implementation would make: retain existing value normalization, but use the cached resolved `path` rather than the leaf’s literal attribute.

The outgoing leaf-frame shape is clear. The remaining routing check is duplicate local names: an incoming `state.title` update must not also reach `state.settings.title`.

In [ ]:
r = await js_eval_a(r"""
const getData = htmx._('getInternalData')

const syncPath = el => {
  const data = getData(el)
  if (data.syncPath) return data.syncPath

  const parts = []
  for (let o=el; o; o=o.parentElement) {
    const part = o.getAttribute('hx-sync')
    if (part) parts.push(part)
  }

  return data.syncPath = parts.reverse().join('.')
}

const root = document.createElement('div')
root.setAttribute('hx-sync', 'state')
root.innerHTML = `
  <input hx-sync="title">
  <div hx-sync="settings"><input hx-sync="title"></div>
`

const els = [...root.querySelectorAll('input')]
return els.map(el => ({
  path:syncPath(el),
  match:syncPath(el)=='state.title',
}))
""")

test_eq(r.result, [
    dict(path='state.title', match=True),
    dict(path='state.settings.title', match=False),
])

This confirms that resolved full paths, rather than literal leaf attributes, give exact routing in both directions.

The browser-side exploration is sufficient to define the prospective composition contract:

```text
non-control hx-sync elements = path scopes
supported controls           = leaf bindings
cached leaf path             = dot-joined scope chain and leaf segment
```

It establishes:

- `state` + `count` resolves to `state.count`;
- nested scopes resolve naturally;
- cached paths make browser events cheap;
- exact full-path matching avoids collisions such as `state.title` versus `state.settings.title`;
- a `state` snapshot can fan out to descendant leaves using relative paths;
- missing snapshot fields leave existing control values unchanged;
- changing a scope attribute in place requires recompilation or subtree replacement.

The remaining unresolved question is kernel-side **root-frame semantics**. The current receiver only accepts leaf assignments such as `state.count`; a root frame such as `path='state'` needs an explicit rule before this becomes a real whole-state sync implementation.

The root-frame rule can be deliberately asymmetric:

```text
leaf frame:  state.count = 17
→ browser or kernel may originate it
→ receiver assigns one kernel attribute

root frame:  state = {count:17, title:'revised'}
→ kernel-originated snapshot only
→ no receiver assignment
→ browser fans it out to descendant leaf controls
```

That avoids deciding whether a root frame should replace `state`, mutate it recursively, preserve object identity, or validate its shape. It also matches the DOM model: scope containers establish paths but do not themselves produce control events.

So a later public API could keep the distinction explicit:

```python
await set_sync('state.count', 17)
await sync_snapshot('state', state)
```

`set_sync` remains a leaf assignment plus broadcast. `sync_snapshot` only serializes and sends a root frame; it never calls `recv_sync`.

The next implementation dialog can first add `sync_snapshot`, then extend browser `sync.apply` so a root frame fans out through the cached descendant paths.

## other bridges

Now I'd like to consider a possible bridge / sync refactor.

The "bridge" concept is an old aspiration of mine. I've implemented several interactions with different architectures. The previous to this Solveit focused version was using `ipywidgets` / `anywidgets` as the messaging transport in Jupyter-ish notebooks, but the underlying ideas were very similar.

I want you to look at that implementation and then discuss similarities, differences and lessons.

Take a look at `bridget` at `https://github.com/civvic/bridget`. Is a `nbdev` project. In `bridget/js` you can find several JS sources implementing the `bridge` and several plugins and helpers.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The JavaScript core is more ambitious than the current Solveit bridge: it combines an anywidget model transport, context-routed handlers, dynamic ES-module/link loading, logging, and an HTMX bootstrap plugin. `bridget.js` then adapts intercepted HTMX/XHR-style requests into request/response messages; `commander.js` uses a shared mutation-observer manager to process newly added nodes. I’ll inspect the Python transport and the remaining browser helpers to identify which ideas are architectural rather than incidental.

I reviewed the `bridget` JavaScript core and its adjacent modules. Its underlying idea is recognizably the same as this bridge, but it accumulated more host integration around the transport.

### Strong similarities

Both designs have the same basic layers:

```text
browser event
→ semantic frame
→ kernel transport
→ named handler
→ browser update or rendered response
```

- Both use a browser-global `bridge`.
- Both support named plugin initialization with cleanup on replacement.
- Both route inbound messages by a namespace: `ctx` in Bridget; `key` in the Solveit bridge.
- Both treat HTMX as an adapter layered above the base transport.
- Both aim to make notebook/document output an active interactive surface rather than a passive rendering.

Bridget’s `bridge.on(ctx, fn)`/`off(ctx)` is the direct predecessor of the current kernel `handlers[key]` dispatcher. Its plugin lifecycle is also closely related to the current `addPlugin`/named-cleanup arrangement.

### Main difference: scope discipline

Bridget’s `bridge.js` is roughly four systems combined:

1. anywidget model transport and lifecycle;
2. context-based message routing;
3. dynamic ESM and FastHTML-link loading;
4. logging, HTMX loading, and default plugins.

It also carries an RPC-like `invoke` path, model state, module cache, loader commands, and plugin initialization notifications.

The current bridge is far narrower:

```text
Channel connection
+ JSON send/receive
+ local events
+ cleanup/plugin lifecycle
```

That is a better boundary for Solveit. `Channel` already supplies the transport. `setup_mod` already supplies a module-installation path. Dialog APIs already supply persistence. Pulling those concerns into `bridge` would recreate Bridget’s broad central object without gaining much.

### Transport difference

Bridget uses an **anywidget model**:

```text
browser model.send(...)
↔ Jupyter comm
↔ Python widget/model
```

The current implementation uses a **named WebSocket Channel**:

```text
browser WebSocket
↔ /wsx?chan=...
↔ Channel.events / Channel.send
```

The semantic design transfers well; the transport-specific lifecycle does not.

Bridget’s `setModel` explicitly closes the old model, removes its message listener, cleans up plugins, and initializes default model-bound plugins on the replacement model. The Solveit analogue will eventually need an explicit policy for:

- browser reconnect;
- duplicate browser connections;
- channel replacement;
- whether plugins replay current state after connection opens.

Those are deferred correctly for now. They should be designed as `Channel` lifecycle policy, not imported wholesale from anywidget.

### Routing and protocol lessons

Bridget frames generally use fields such as:

```javascript
{ctx, kind, cmd, args, msg_id}
```

The current sync frames use:

```javascript
{key:'sync', op:'set', path, value, raw}
```

The current schema is better for declarative state synchronization: `key` selects the semantic subsystem; `op`, `path`, and `value` express the operation directly.

Bridget demonstrates one useful future extension: **correlation IDs**. Its request/response layer assigns `req_id` and stores callbacks in a pending map. That is valuable only when Solveit needs a true browser RPC:

```text
kernel asks browser to measure/select/read something
→ browser responds asynchronously
→ caller awaits the matching reply
```

Do not add that machinery to ordinary sync frames. If needed, make it a small `rpc` plugin or a Channel-level request helper with explicit request IDs and timeout/cancellation behavior.

### Plugin lessons

Bridget’s plugin API supports a cleanup callback and optional exposed API object:

```javascript
[cleanup, api]
```

The current API already has the most important property: deterministic replacement cleanup.

One improvement worth considering later is an explicit convention for exposed plugin APIs:

```javascript
bridge.addPlugin('sync', bridge.initSync)
bridge.sync.send(...)
bridge.sync.apply(...)
```

You already have this in practice. It is clearer than Bridget’s convention because the API is assigned directly and each plugin owns one visible namespace.

A difference worth preserving: Bridget’s `msgHub` is a `Map`, so each `ctx` has only one receiver. The current `bridge.on('msg', ...)` event fan-out permits independent browser-local observers, while the kernel has a single dispatcher to avoid competing `Channel.events` consumers. That is a sound asymmetry:

```text
browser: many observers are harmless
kernel Channel queue: one consumer is necessary
```

### HTMX and mutation-observer lesson

Bridget’s `commander.js` maintains a shared `MutationObserver` manager. It watches added DOM nodes and calls `htmx.process(node)` for selected targets. That made sense for its serverless HTMX rendering environment, where inserted notebook output needed explicit HTMX processing.

For this bridge, the existing `htmx:afterSwap` adapter is preferable for synchronization:

```text
HTMX reports a completed swap
→ adapter receives the actual swapped root
→ adapter finds hx-sync controls
→ sync.send handles the normal semantic route
```

It is more precise than observing all document mutations and avoids treating arbitrary DOM changes as a synchronization event.

The earlier Bridget work does still support one future principle: if Solveit gains a genuinely host-level DOM insertion mechanism that does **not** emit an HTMX lifecycle event, use one shared observer manager rather than allowing each control plugin to install its own `MutationObserver`.

### Dynamic loading lesson

Bridget’s `brdimport.js` dynamically resolves modules, asks Python for module source through an invoke call when necessary, imports inline ESM, caches it, and records loaded module names in the widget model.

That capability is substantial, but Solveit already has a cleaner equivalent boundary:

```text
exported JS source
→ setup_mod(...)
→ browser module installation
```

If dynamic plugin loading becomes necessary, it should sit beside `setup_mod` as a **module-loader layer**, not inside `bridge`. The bridge should not need to know URLs, blobs, imports, package names, or notebook exports.

### Document-state lesson

Bridget’s notebook-state helpers and `nbstate.js` are particularly relevant philosophically. They connect browser-side live notebook inspection to Python-side state updates. That rhymes with the current goal:

```text
live UI interaction
→ semantic kernel state
→ durable dialog projection
→ later human/model context
```

But Solveit’s durable dialog message model is stronger and more explicit than Bridget’s browser-notebook introspection. The lesson is not to transplant Bridget’s notebook-state machinery. It is to retain the separation already established here:

```text
bridge / Channel        = live transport
sync                    = control semantics
projection              = durable dialog update
dialog model            = durable truth
```

### Refactor direction

The current bridge should stay small. A useful eventual structure would be:

```text
bridge
  Channel lifecycle, JSON ingress/egress, local event bus, plugin cleanup

sync
  control normalization, cached resolved paths, leaf update/snapshot fan-out

htmx
  HTMX lifecycle adaptation and perhaps compiled element metadata

rpc                 # only if a real request/reply need appears
  correlation IDs, timeout, cancellation

projection
  sync-path → durable dialog artifact rules

module loader       # only if setup_mod becomes insufficient
  installation/reload of browser modules
```

The main lesson from Bridget is that the original aspiration was right: browser, kernel, and notebook/document can form one interactive surface. The caution is architectural: its older implementation put transport, plugin loading, host adaptation, request interception, logging, and document integration in one unusually capable `bridge`. This Solveit version should preserve the aspiration while keeping those responsibilities as separate layers.

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 313176, "completion_tokens": 3438, "total_tokens": 316614, "cached_tokens": 164352, "reasoning_tokens": 1030, "cost": 0.049554900000000006}
```


I like the plugin model better. IN our current implmentation we need two calls and besides that augment `bridge` object with several `init*` functions. I think is cleaner for each plugin to conform to a minimal API (init, for now) and then have some plungin introspection or directly access the plugin with `bridge.<plugin name>`.

I also like a lot the `observer` plugin, but probably here in Solveit it won't be necessary for now.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Agreed. The current pattern splits one conceptual operation:

```javascript
bridge.initSync = bridge => { ... }
bridge.addPlugin('sync', bridge.initSync)
```

A plugin object gives it a single, inspectable unit instead:

```javascript
const sync = {
  init(bridge) {
    const api = {send, apply}
    return {api, cleanup}
  }
}

bridge.use('sync', sync)
```

The bridge manager can then own replacement, cleanup, and exposed APIs:

```javascript
bridge.plugins ||= {}

bridge.use = (nm, plugin) => {
  bridge.plugins[nm]?.cleanup()
  const {api={},cleanup=()=>{}} = plugin.init(bridge) || {}
  bridge.plugins[nm] = {plugin,api,cleanup}
  bridge[nm] = api
  return api
}

bridge.unuse = nm => {
  bridge.plugins[nm]?.cleanup()
  delete bridge.plugins[nm]
  delete bridge[nm]
}
```

That yields both forms of introspection:

```javascript
bridge.sync.send(el, event)
bridge.sync.apply(msg)

bridge.plugins.sync.plugin
bridge.plugins.sync.api
```

The lifecycle contract stays minimal:

```text
plugin.init(bridge)
→ {api, cleanup}
```

- `api` is the deliberate public surface, exposed as `bridge.<name>`.
- `cleanup` tears down listeners, subscriptions, and plugin-local DOM state.
- `bridge.use` owns replacement ordering and removes the old public API.
- Plugins no longer need to attach `initSync`, `initHtmx`, and similar installation functions onto `bridge`.

For this project, `sync` and `htmx` are natural first conversions:

```javascript
bridge.use('sync', syncPlugin)
bridge.use('htmx', htmxPlugin)
```

An observer plugin fits the same model later:

```javascript
bridge.use('observer', observerPlugin)
```

but should remain absent until there is a concrete non-HTMX DOM-insertion path that needs shared mutation observation. The current `htmx:afterSwap` adapter is more precise for the behavior it covers.

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 78635, "completion_tokens": 962, "total_tokens": 79597, "cached_tokens": 64896, "reasoning_tokens": 516, "cost": 0.012516750000000002}
```


Agreed.

## browser bridge v1

Start v1 by replacing only the generic plugin manager. Keep `addCleanup` for non-plugin resources such as the Channel; convert `sync` and `htmx` only after this contract passes.

In [ ]:
await js_eval_a("""bridge.plugins ||= {}

bridge.unuse = nm => {
  const p = bridge.plugins[nm]
  if (!p) return
  p.cleanup()
  delete bridge.plugins[nm]
  if (bridge[nm]===p.api) delete bridge[nm]
}

bridge.use = (nm, plugin) => {
  bridge.unuse(nm)
  const {api={},cleanup=()=>{}} = plugin.init(bridge) || {}
  bridge.plugins[nm] = {plugin,api,cleanup}
  bridge[nm] = api
  return api
}""")

```python
{'data_id': 'ca7c093a-dc50-40f1-9883-4fa98f2621e4'}
```

Test replacement, public API exposure, and cleanup. Expected result:

In [ ]:
res = await js_eval_a("""
const seen = []
const plugin = tag => ({
  init: bridge => ({
    api: {tag},
    cleanup: () => seen.push(tag),
  }),
})

const first = bridge.use('test', plugin('first'))
const second = bridge.use('test', plugin('second'))
const before = {first:first.tag, second:second.tag, live:bridge.test.tag, names:Object.keys(bridge.plugins)}

bridge.unuse('test')

return {before, seen, live:bridge.test, names:Object.keys(bridge.plugins)}""")

test_eq(res.result, {
  'before': { 'first': 'first', 'live': 'second', 'names': ['test'], 'second': 'second'}, 
  'names': [],
  'seen': ['first', 'second']})

### sync plugin v1

The generic manager passed. `live` is absent from the returned JSON because JavaScript omits properties whose value is `undefined`; the cleanup order and plugin registry match the intended contract.

Convert `sync` next. Remove its legacy `addPlugin` installation before using the new manager; otherwise its old delegated listeners would remain active alongside the new plugin.

In [ ]:
await add_js_msg('_sync_js', i_collapsed=True);

In [ ]:
_sync_js = r'''const syncPlugin = {
  init(bridge) {
    const api = {}

    api.val = el => {
      if (el instanceof HTMLInputElement) {
        if (el.type=='checkbox') return el.checked
        if (['range', 'number'].includes(el.type)) {
          const v = el.valueAsNumber
          return Number.isNaN(v) ? null : v
        }
        return el.value
      }
      if (el instanceof HTMLSelectElement) return el.multiple ? [...el.selectedOptions].map(o => o.value) : el.value
      if (el instanceof HTMLTextAreaElement) return el.value
    }

    api.send = (el, event) => {
      const path = el.getAttribute('hx-sync'), value = api.val(el)
      if (!path || value===undefined) return
      const msg = {key:'sync', op:'set', path, value, raw:el.value}
      bridge.recv(el, msg, event)
      bridge.send(msg)
      return msg
    }

    api.apply = msg => {
      if (msg.key!='sync' || msg.op!='set') return
      for (const el of document.querySelectorAll('[hx-sync]')) {
        if (el.getAttribute('hx-sync')!=msg.path) continue
        if (el instanceof HTMLInputElement && el.type=='checkbox') el.checked = msg.value
        else if (el instanceof HTMLSelectElement && el.multiple) {
          for (const o of el.options) o.selected = msg.value.includes(o.value)
        } else if (el instanceof HTMLInputElement && el.type=='number' && msg.value==null) el.value = ''
        else if (el instanceof HTMLInputElement || el instanceof HTMLSelectElement || el instanceof HTMLTextAreaElement) el.value = msg.value
      }
    }

    const change = e => api.send(e.target, e)
    document.addEventListener('input', change)
    document.addEventListener('change', change)
    const off = bridge.on('msg', ({msg}) => api.apply(msg))

    return {api, cleanup: () => {
      document.removeEventListener('input', change)
      document.removeEventListener('change', change)
      off()
    }}
  }
}

bridge.cleanup('plugin:sync')
bridge.use('sync', syncPlugin)'''
#| linkedto: 6fc8a606

In [ ]:
await setup_mod('sync', _sync_js, True)

This preserves the public API exactly:

```javascript
bridge.sync.send(el, event)
bridge.sync.apply(msg)
```

but moves the implementation and lifecycle under the inspectable plugin record:

```javascript
bridge.plugins.sync.plugin
bridge.plugins.sync.api
```

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The new `sync` plugin is installed and exposes the existing API. Test replacement now: it must leave exactly one delegated listener and retain the public/plugin registry relationship.

```python
r = await js_eval_a(r"""
bridge.use('sync', syncPlugin)
bridge.msgs = []

const el = document.getElementById('sync-title')
el.value = 'unprojected'
el.dispatchEvent(new Event('input', {bubbles:true}))

return {
  n: bridge.msgs.filter(msg => msg.path=='state.title').length,
  api: bridge.plugins.sync.api===bridge.sync,
  plugin: bridge.plugins.sync.plugin===syncPlugin,
}
""")

test_eq(r.result, dict(n=1, api=True, plugin=True))
```

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 81729, "completion_tokens": 674, "total_tokens": 82403, "cached_tokens": 44416, "reasoning_tokens": 516, "cost": 0.012764850000000001}
```


`syncPlugin` was installed in a module, is not available globally.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Correct. `syncPlugin` is module-local; use the installed plugin object from the registry before replacing it:

```python
r = await js_eval_a(r"""
const plugin = bridge.plugins.sync.plugin
bridge.use('sync', plugin)
bridge.msgs = []

const el = document.getElementById('sync-title')
el.value = 'unprojected'
el.dispatchEvent(new Event('input', {bubbles:true}))

return {
  n: bridge.msgs.filter(msg => msg.path=='state.title').length,
  api: bridge.plugins.sync.api===bridge.sync,
  plugin: bridge.plugins.sync.plugin===plugin,
}
""")

test_eq(r.result, dict(n=1, api=True, plugin=True))
```

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 81955, "completion_tokens": 387, "total_tokens": 82342, "cached_tokens": 81280, "reasoning_tokens": 234, "cost": 0.012583500000000004}
```


In [ ]:
r = await js_eval_a(r"""
const plugin = bridge.plugins.sync.plugin
bridge.use('sync', plugin)
bridge.msgs = []

const el = document.getElementById('sync-title')
el.value = 'unprojected'
el.dispatchEvent(new Event('input', {bubbles:true}))

return {
  n: bridge.msgs.filter(msg => msg.path=='state.title').length,
  api: bridge.plugins.sync.api===bridge.sync,
  plugin: bridge.plugins.sync.plugin===plugin,
}
""")

test_eq(r.result, dict(n=1, api=True, plugin=True))

Hmmm, something weird here. Perhaps we should distinguish between loading/registering vs suing a plugin (equivalent to HTMX `<script type="text/javascript" src="vendor/ws-2.0.3.js"></script>` and `<div ws-connect="..." hx-ext="ws" id="ws-div"></div>`).

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Exactly. The current module conflates two separate lifecycle stages:

```text
setup_mod('sync', _sync_js)
→ evaluate module
→ define syncPlugin
→ immediately bridge.use('sync', syncPlugin)
```

That is why the plugin definition is module-local but only recoverable indirectly from the active record. It also makes “load this capability” indistinguishable from “activate it in this document.”

Use three explicit states instead:

```text
registered plugin definition   bridge.plugin_defs.sync
active plugin instance         bridge.plugins.sync
active public API              bridge.sync
```

A small generic manager could be:

```javascript
bridge.plugin_defs ||= {}
bridge.plugins ||= {}

bridge.register = (nm, plugin) => bridge.plugin_defs[nm] = plugin

bridge.unuse = nm => {
  const p = bridge.plugins[nm]
  if (!p) return
  p.cleanup()
  delete bridge.plugins[nm]
  if (bridge[nm]===p.api) delete bridge[nm]
}

bridge.use = (nm, opts={}) => {
  bridge.unuse(nm)
  const plugin = bridge.plugin_defs[nm]
  if (!plugin) throw new Error(`Unknown bridge plugin: ${nm}`)
  const {api={},cleanup=()=>{}} = plugin.init(bridge, opts) || {}
  bridge.plugins[nm] = {plugin,api,cleanup,opts}
  bridge[nm] = api
  return api
}
```

Then the installed sync module only **registers** its definition:

```javascript
const syncPlugin = {
  init(bridge, opts) {
    // current sync implementation
  }
}

bridge.register('sync', syncPlugin)
```

Activation becomes a distinct, document-level decision:

```javascript
bridge.use('sync')
```

That gives the desired analogy:

```text
<script src="sync-plugin.js">  → module loaded and plugin registered
hx-ext="sync"                  → plugin activated for this document
```

For now, `bridge.use('sync')` can remain a global document activation. Later, its `opts` argument can support scoped roots or per-instance configuration without changing the registration model.

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 82386, "completion_tokens": 967, "total_tokens": 83353, "cached_tokens": 81280, "reasoning_tokens": 516, "cost": 0.013083150000000002}
```


Don't know about that. If we stablish (as I think we should in 2026) extensions as ES modules we just need `import` to get the definition (if exported, that is, unlike our version above). So we can maintain our current plugin registry and lifecycle, no need to store the defs.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Agreed. I was adding a registry to solve a problem that ES modules already solve.

Keep only the **active-instance registry**:

```text
imported module export  = plugin definition
bridge.plugins[name]   = active instance and cleanup
bridge[name]           = active public API
```

So the module should export its definition without activating itself:

```javascript
export const syncPlugin = {
  init(bridge) {
    // return {api, cleanup}
  }
}
```

Then the importing code holds the definition and explicitly activates it:

```javascript
import {syncPlugin} from './sync.js'

bridge.use('sync', syncPlugin)
```

`bridge.use` remains responsible only for lifecycle:

```javascript
bridge.use = (nm, plugin) => {
  bridge.unuse(nm)
  const {api={},cleanup=()=>{}} = plugin.init(bridge) || {}
  bridge.plugins[nm] = {plugin,api,cleanup}
  bridge[nm] = api
  return api
}
```

That gives the desired separation without a duplicate definition registry:

```text
ESM import       → makes an extension available to code
bridge.use(...)  → activates one instance in the live document
bridge.unuse(...)→ tears that instance down
```

It also fixes the earlier test naturally: import `syncPlugin` in the code that wants to reinstall it, rather than recovering it from `bridge.plugins.sync.plugin`.

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 82956, "completion_tokens": 420, "total_tokens": 83376, "cached_tokens": 82304, "reasoning_tokens": 118, "cost": 0.0127584}
```


.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The next change is to make the sync module define capability without activating it.

Change its declaration and tail to:

```javascript
export const syncPlugin = {
  init(bridge) {
    // existing implementation
  }
}
```

Remove these activation-side effects:

```javascript
bridge.cleanup('plugin:sync')
bridge.use('sync', syncPlugin)
```

Activation then belongs in a separate document/bootstrap module, which imports the definition and decides to use it:

```javascript
import {syncPlugin} from './sync.js'

bridge.use('sync', syncPlugin)
```

This gives the intended separation:

```text
ESM export/import = capability available
bridge.use(...)   = capability active
```

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 83290, "completion_tokens": 664, "total_tokens": 83954, "cached_tokens": 82304, "reasoning_tokens": 516, "cost": 0.0129915}
```


In [ ]:
await add_js_msg('_sync_js', i_collapsed=True);

In [ ]:
_sync_js = r'''export const syncPlugin = {
  init(bridge) {
    const api = {}

    api.val = el => {
      if (el instanceof HTMLInputElement) {
        if (el.type=='checkbox') return el.checked
        if (['range', 'number'].includes(el.type)) {
          const v = el.valueAsNumber
          return Number.isNaN(v) ? null : v
        }
        return el.value
      }
      if (el instanceof HTMLSelectElement) return el.multiple ? [...el.selectedOptions].map(o => o.value) : el.value
      if (el instanceof HTMLTextAreaElement) return el.value
    }

    api.send = (el, event) => {
      const path = el.getAttribute('hx-sync'), value = api.val(el)
      if (!path || value===undefined) return
      const msg = {key:'sync', op:'set', path, value, raw:el.value}
      bridge.recv(el, msg, event)
      bridge.send(msg)
      return msg
    }

    api.apply = msg => {
      if (msg.key!='sync' || msg.op!='set') return
      for (const el of document.querySelectorAll('[hx-sync]')) {
        if (el.getAttribute('hx-sync')!=msg.path) continue
        if (el instanceof HTMLInputElement && el.type=='checkbox') el.checked = msg.value
        else if (el instanceof HTMLSelectElement && el.multiple) {
          for (const o of el.options) o.selected = msg.value.includes(o.value)
        } else if (el instanceof HTMLInputElement && el.type=='number' && msg.value==null) el.value = ''
        else if (el instanceof HTMLInputElement || el instanceof HTMLSelectElement || el instanceof HTMLTextAreaElement) el.value = msg.value
      }
    }

    const change = e => api.send(e.target, e)
    document.addEventListener('input', change)
    document.addEventListener('change', change)
    const off = bridge.on('msg', ({msg}) => api.apply(msg))

    return {api, cleanup: () => {
      document.removeEventListener('input', change)
      document.removeEventListener('change', change)
      off()
    }}
  }
}'''
#| linkedto: dea674d5

In [ ]:
await setup_mod('sync', _sync_js, True)

('sync_3',
 {'data_id': 'f8982805-fd97-4d0c-bb35-ea8dc77fd79a',
  'result': {'syncPlugin': {}}})

In [ ]:
await js_eval_a(r"""bridge.unuse('sync')""")
test_is(await hx.eval_a('window.bridge.sync===undefined'), True)

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

`sync` is now loaded but inactive, as intended. Activate the exported definition explicitly, using the module name returned by `setup_mod`:

```python
r = await js_eval_a(r"""
const {syncPlugin} = await import('sync_3')
bridge.use('sync', syncPlugin)
return {
  active: bridge.plugins.sync.plugin===syncPlugin,
  api: typeof bridge.sync.send=='function',
}
""")

test_eq(r.result, dict(active=True, api=True))
```

This establishes the separation:

```text
setup_mod(...) / ESM import  → plugin definition available
bridge.use(...)              → plugin active in this document
```

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 84582, "completion_tokens": 656, "total_tokens": 85238, "cached_tokens": 8576, "reasoning_tokens": 516, "cost": 0.013179300000000001}
```


In [ ]:
r = await js_eval_a(r"""
const {syncPlugin} = await import('/esm/sync?v=3')
bridge.use('sync', syncPlugin)
return {
  active: bridge.plugins.sync.plugin===syncPlugin,
  api: typeof bridge.sync.send=='function',
}
""")

test_eq(r.result, dict(active=True, api=True))

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Test that the explicitly activated ESM plugin retains the existing browser-to-kernel contract:

```python
state.title = 'before module test'

await js_eval_a(r"""
bridge.msgs = []
const el = document.getElementById('sync-title')
el.value = 'from module plugin'
el.dispatchEvent(new Event('input', {bubbles:true}))
""")

await wait_for_a(lambda: state.title=='from module plugin')
test_eq(bridge.plugins.sync.api, bridge.sync)
test_eq(state.title, 'from module plugin')
```

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 84678, "completion_tokens": 338, "total_tokens": 85016, "cached_tokens": 8576, "reasoning_tokens": 223, "cost": 0.0129552}
```


In [ ]:
async def recv_sync(msg):
    if msg.get('op')!='set': return
    root,*ks = msg.path.split('.')
    if not ks: return
    o = find_var(root)
    for k in ks[:-1]: o = getattr(o, k)
    setattr(o, ks[-1], msg.value)
    await apply_sync_proj(msg.path, msg.value)
    return msg.value

handlers['sync'] = recv_sync

In [ ]:
bridge_task = await start_bridge(ch)
test_is(bridge_task.done(), False)

In [ ]:
state.title = 'before module test'

await js_eval_a(r"""
bridge.msgs = []
const el = document.getElementById('sync-title')
el.value = 'from module plugin'
el.dispatchEvent(new Event('input', {bubbles:true}))
""")

await wait_for_a(lambda: state.title=='from module plugin')
test_is(await hx.eval_a('bridge.plugins.sync.api===bridge.sync'), True)
test_eq(state.title, 'from module plugin')

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The ESM-loaded plugin now passes its normal browser → kernel path. Confirm replacement still leaves one listener:

```python
state.title = 'before replacement test'

r = await js_eval_a(r"""
const {syncPlugin} = await import('/esm/sync_3')
bridge.use('sync', syncPlugin)
bridge.msgs = []

const el = document.getElementById('sync-title')
el.value = 'after replacement test'
el.dispatchEvent(new Event('input', {bubbles:true}))

return {
  n: bridge.msgs.filter(msg => msg.path=='state.title').length,
  api: bridge.plugins.sync.api===bridge.sync,
  plugin: bridge.plugins.sync.plugin===syncPlugin,
}
""")

await wait_for_a(lambda: state.title=='after replacement test')
test_eq(r.result, dict(n=1, api=True, plugin=True))
```

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 86407, "completion_tokens": 588, "total_tokens": 86995, "cached_tokens": 85376, "reasoning_tokens": 406, "cost": 0.01340205}
```


In [ ]:
state.title = 'before replacement test'

r = await js_eval_a(r"""
const {syncPlugin} = await import('/esm/sync?v=3')
bridge.use('sync', syncPlugin)
bridge.msgs = []

const el = document.getElementById('sync-title')
el.value = 'after replacement test'
el.dispatchEvent(new Event('input', {bubbles:true}))

return {
  n: bridge.msgs.filter(msg => msg.path=='state.title').length,
  api: bridge.plugins.sync.api===bridge.sync,
  plugin: bridge.plugins.sync.plugin===syncPlugin,
}
""")

await wait_for_a(lambda: state.title=='after replacement test')
test_eq(r.result, dict(n=1, api=True, plugin=True))

### htmx plugin v1

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The ESM `sync` plugin passes activation, browser-to-kernel delivery, and replacement without duplicate listeners.

Convert the HTMX adapter to the same definition-only module next:

```javascript
export const htmxPlugin = {
  init(bridge) {
    const afterSwap = e => {
      const root = e.detail.elt
      const els = [root, ...(root.querySelectorAll?.('[hx-sync]') || [])].filter(el => el?.matches?.('[hx-sync]'))
      for (const el of els) bridge.sync?.send(el, e)
    }

    document.body.addEventListener('htmx:afterSwap', afterSwap)
    return {api:{}, cleanup:() => document.body.removeEventListener('htmx:afterSwap', afterSwap)}
  }
}
```

It should not call `bridge.use(...)`. A bootstrap cell can later import and activate it:

```javascript
const {htmxPlugin} = await import('/esm/htmx_...')
bridge.use('htmx', htmxPlugin)
```

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 86808, "completion_tokens": 419, "total_tokens": 87227, "cached_tokens": 8576, "reasoning_tokens": 197, "cost": 0.01333545}
```


In [ ]:
await js_eval_a(r"""bridge.cleanup('plugin:htmx')""")

```python
{'data_id': 'c26116d3-9e3f-4758-90d1-09d330439d4a'}
```

In [ ]:
await add_js_msg('_htmx_js', i_collapsed=True, exported=True);

In [ ]:
_htmx_js = r'''export const htmxPlugin = {
  init(bridge) {
    const afterSwap = e => {
      const root = e.detail.elt
      const els = [root, ...(root.querySelectorAll?.('[hx-sync]') || [])].filter(el => el?.matches?.('[hx-sync]'))
      for (const el of els) bridge.sync?.send(el, e)
    }

    document.body.addEventListener('htmx:afterSwap', afterSwap)
    return {api:{}, cleanup:() => document.body.removeEventListener('htmx:afterSwap', afterSwap)}
  }
}'''
#| linkedto: c21eed86

In [ ]:
await setup_mod('htmx', _htmx_js, True)

('htmx_2',
 {'data_id': '8c1b02d6-04c4-4ea4-a487-520af7629fd7',
  'result': {'htmxPlugin': {}}})

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Activate the ESM definition explicitly, then verify the active record and public API:

```python
r = await js_eval_a(r"""
const {htmxPlugin} = await import('/esm/htmx?v=2')
bridge.use('htmx', htmxPlugin)

return {
  active: bridge.plugins.htmx.plugin===htmxPlugin,
  api: bridge.plugins.htmx.api===bridge.htmx,
}
""")

test_eq(r.result, dict(active=True, api=True))
```

`bridge.htmx` is the plugin’s deliberately empty public API for now; `window.htmx` remains the actual HTMX library.

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 86477, "completion_tokens": 445, "total_tokens": 86922, "cached_tokens": 3456, "reasoning_tokens": 310, "cost": 0.013305300000000003}
```


In [ ]:
r = await js_eval_a(r"""
const {htmxPlugin} = await import('/esm/htmx?v=2')
bridge.use('htmx', htmxPlugin)

return {
  active: bridge.plugins.htmx.plugin===htmxPlugin,
  api: bridge.plugins.htmx.api===bridge.htmx,
}
""")

test_eq(r.result, dict(active=True, api=True))

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The HTMX plugin is active through the new lifecycle manager. Re-run one established swap test to confirm the ESM activation preserved its semantic contract:

```python
state.count = 13

r = await js_eval_a(r"""
bridge.msgs = []
const arena = document.getElementById('bridge-arena01')
const root = document.createElement('input')
root.type = 'range'
root.value = 25
root.setAttribute('hx-sync', 'state.count')

arena.append(root)
arena.dispatchEvent(new CustomEvent('htmx:afterSwap', {bubbles:true, detail:{elt:root}}))
root.remove()

return bridge.msgs.filter(msg => msg.path=='state.count')
""")

await wait_for_a(lambda: state.count==25)
test_eq(r.result, [dict(key='sync', op='set', path='state.count', value=25, raw='25')])
test_eq(state.count, 25)
```

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 86759, "completion_tokens": 414, "total_tokens": 87173, "cached_tokens": 86400, "reasoning_tokens": 215, "cost": 0.013324350000000002}
```


In [ ]:
state.count = 13

r = await js_eval_a(r"""
bridge.msgs = []
const arena = document.getElementById('bridge-arena01')
const root = document.createElement('input')
root.type = 'range'
root.value = 25
root.setAttribute('hx-sync', 'state.count')

arena.append(root)
arena.dispatchEvent(new CustomEvent('htmx:afterSwap', {bubbles:true, detail:{elt:root}}))
root.remove()

return bridge.msgs.filter(msg => msg.path=='state.count')
""")

await wait_for_a(lambda: state.count==25)
test_eq(r.result, [dict(key='sync', op='set', path='state.count', value=25, raw='25')])
test_eq(state.count, 25)

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The ESM HTMX plugin preserves the established swap route. Test replacement once, to confirm it also leaves one `htmx:afterSwap` listener:

```python
state.count = 13

r = await js_eval_a(r"""
const {htmxPlugin} = await import('/esm/htmx?v=2')
bridge.use('htmx', htmxPlugin)
bridge.msgs = []

const arena = document.getElementById('bridge-arena01')
const root = document.createElement('input')
root.type = 'range'
root.value = 24
root.setAttribute('hx-sync', 'state.count')

arena.append(root)
arena.dispatchEvent(new CustomEvent('htmx:afterSwap', {bubbles:true, detail:{elt:root}}))
root.remove()

return {
  n: bridge.msgs.filter(msg => msg.path=='state.count').length,
  api: bridge.plugins.htmx.api===bridge.htmx,
  plugin: bridge.plugins.htmx.plugin===htmxPlugin,
}
""")

await wait_for_a(lambda: state.count==24)
test_eq(r.result, dict(n=1, api=True, plugin=True))
```

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 87269, "completion_tokens": 352, "total_tokens": 87621, "cached_tokens": 8576, "reasoning_tokens": 110, "cost": 0.013354350000000003}
```


In [ ]:
state.count = 13

r = await js_eval_a(r"""
const {htmxPlugin} = await import('/esm/htmx?v=2')
bridge.use('htmx', htmxPlugin)
bridge.msgs = []

const arena = document.getElementById('bridge-arena01')
const root = document.createElement('input')
root.type = 'range'
root.value = 24
root.setAttribute('hx-sync', 'state.count')

arena.append(root)
arena.dispatchEvent(new CustomEvent('htmx:afterSwap', {bubbles:true, detail:{elt:root}}))
root.remove()

return {
  n: bridge.msgs.filter(msg => msg.path=='state.count').length,
  api: bridge.plugins.htmx.api===bridge.htmx,
  plugin: bridge.plugins.htmx.plugin===htmxPlugin,
}
""")

await wait_for_a(lambda: state.count==24)
test_eq(r.result, dict(n=1, api=True, plugin=True))

### bridge v1

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The `sync` and `htmx` ESM plugins now pass activation, normal behavior, and replacement tests without duplicate listeners.

The remaining v1 step is to make the generic manager durable in the exported `bridge` module. Keep `addCleanup` for resources such as the Channel, but replace the legacy plugin API:

```javascript
bridge.addPlugin = (nm, init) => bridge.addCleanup(`plugin:${nm}`, init(bridge) || (() => {}))
```

with:

```javascript
bridge.plugins ||= {}

bridge.unuse = nm => {
  const p = bridge.plugins[nm]
  if (!p) return
  p.cleanup()
  delete bridge.plugins[nm]
  if (bridge[nm]===p.api) delete bridge[nm]
}

bridge.use = (nm, plugin) => {
  bridge.unuse(nm)
  const {api={},cleanup=()=>{}} = plugin.init(bridge) || {}
  bridge.plugins[nm] = {plugin,api,cleanup}
  bridge[nm] = api
  return api
}
```

That leaves the browser core with a clean division:

```text
bridge.addCleanup(...)  → generic named resources, including Channel teardown
bridge.use(...)         → activate an imported ESM plugin
bridge.unuse(...)       → remove its listeners and public API
bridge.plugins[name]    → inspect the active plugin instance
bridge[name]            → use its intentional public API
```

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 87780, "completion_tokens": 672, "total_tokens": 88452, "cached_tokens": 8576, "reasoning_tokens": 369, "cost": 0.013671}
```


In [ ]:
await add_js_msg('_bridge_js', i_collapsed=True);

In [ ]:
_bridge_js = r'''const bridge = globalThis.bridge || {}

bridge.msgs ||= []
bridge.cbs ||= {}
bridge.cleanups ||= {}

bridge.on = (nm, f) => {
  ;(bridge.cbs[nm] ||= []).push(f)
  return () => bridge.off(nm, f)
}

bridge.off = (nm, f) => {
  if (!f) return delete bridge.cbs[nm]
  bridge.cbs[nm] = (bridge.cbs[nm] || []).filter(g => g!==f)
  if (!bridge.cbs[nm].length) delete bridge.cbs[nm]
}

bridge.emit = (nm, data) => (bridge.cbs[nm] || []).map(f => f(data))

bridge.recv = (el, msg, event) => {
  bridge.msgs.push(msg)
  bridge.emit('msg', {el, msg, event})
  return msg
}

bridge.cleanup = nm => {
  const f = bridge.cleanups[nm]
  delete bridge.cleanups[nm]
  return f?.()
}

bridge.addCleanup = (nm, f) => {
  bridge.cleanup(nm)
  bridge.cleanups[nm] = f
  return () => bridge.cleanup(nm)
}

bridge.plugins ||= {}
delete bridge.addPlugin

bridge.unuse = nm => {
  const p = bridge.plugins[nm]
  if (!p) return
  p.cleanup()
  delete bridge.plugins[nm]
  if (bridge[nm]===p.api) delete bridge[nm]
}

bridge.use = (nm, plugin) => {
  bridge.unuse(nm)
  const {api={},cleanup=()=>{}} = plugin.init(bridge) || {}
  bridge.plugins[nm] = {plugin,api,cleanup}
  bridge[nm] = api
  return api
}

bridge.chanUrl = chan => {
  const url = new URL('/wsx', location.href)
  url.searchParams.set('chan', chan)
  url.protocol = location.protocol=='https:' ? 'wss:' : 'ws:'
  return url.href
}

bridge.connect = chan => {
  bridge.cleanup('channel')
  const ws = bridge.ws = new WebSocket(bridge.chanUrl(chan))

  ws.onopen = event => bridge.emit('channel:open', {chan, ws, event})

  ws.onmessage = event => {
    try { bridge.recv(null, JSON.parse(event.data), event) }
    catch (err) { bridge.emit('channel:error', {chan, ws, event, err}) }
  }

  ws.onerror = event => bridge.emit('channel:error', {chan, ws, event})

  ws.onclose = event => {
    if (bridge.ws===ws) bridge.ws = null
    bridge.emit('channel:close', {chan, ws, event})
  }

  bridge.addCleanup('channel', () => {
    if (bridge.ws===ws) bridge.ws = null
    ws.close()
  })

  return ws
}

bridge.send = msg => {
  const ws = bridge.ws
  if (!ws || ws.readyState!=WebSocket.OPEN) return false
  ws.send(JSON.stringify(msg))
  return true
}

globalThis.bridge = bridge
export {bridge}'''
#| linkedto: aac16400

In [ ]:
await setup_mod('bridge', _bridge_js, True)

('bridge_2',
 {'data_id': 'e2027384-7a5a-4ef5-b35e-29f25ba42b7b',
  'result': {'bridge': {'msgs': [{'key': 'sync', 'op': 'set', 'path': 'state.count', 'value': 24, 'raw': '24'}],
    'cbs': {'msg': [None]},
    'cleanups': {},
    'ws': {},
    'plugins': {'sync': {'plugin': {}, 'api': {}},
     'htmx': {'plugin': {}, 'api': {}}},
    'sync': {},
    'htmx': {}}}})

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The updated core preserved the active `sync` and `htmx` instances because it reuses `globalThis.bridge`. Confirm that the exported module, global object, lifecycle API, and active registry are aligned:

```python
r = await js_eval_a(r"""
const {bridge: exported} = await import('/esm/bridge?v=2')

return {
  same: exported===globalThis.bridge,
  use: typeof bridge.use=='function',
  unuse: typeof bridge.unuse=='function',
  active: Object.keys(bridge.plugins).sort(),
}
""")

test_eq(r.result, dict(same=True, use=True, unuse=True, active=['htmx','sync']))
```

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 90126, "completion_tokens": 579, "total_tokens": 90705, "cached_tokens": 8576, "reasoning_tokens": 433, "cost": 0.013953150000000001}
```


In [ ]:
r = await js_eval_a(r"""
const {bridge: exported} = await import('/esm/bridge?v=2')

return {
  same: exported===globalThis.bridge,
  use: typeof bridge.use=='function',
  unuse: typeof bridge.unuse=='function',
  active: Object.keys(bridge.plugins).sort(),
}
""")

test_eq(r.result, dict(same=True, use=True, unuse=True, active=['htmx','sync']))

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The core contract now passes. Separate document activation into one small bootstrap module: it imports capabilities, then explicitly activates them.

```javascript
import {bridge} from '/esm/bridge'
import {syncPlugin} from '/esm/sync'
import {htmxPlugin} from '/esm/htmx'

bridge.use('sync', syncPlugin)
bridge.use('htmx', htmxPlugin)

export {bridge}
```

This establishes the intended lifecycle:

```text
bridge/sync/htmx ESM modules  → definitions available
bootstrap module              → document chooses active extensions
bridge.plugins                → current active instances
bridge.sync / bridge.htmx     → public APIs
```

The bootstrap import also replaces the currently active instances with the exported module definitions, so it becomes the single authoritative activation point.

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 89834, "completion_tokens": 604, "total_tokens": 90438, "cached_tokens": 8576, "reasoning_tokens": 435, "cost": 0.013928100000000002}
```


So now the public API of `bridge` is this:

**debug**
```text
bridge.msgs
```

**message hub & hooks lifecycle**
```text
bridge.cbs
bridge.cleanups
bridge.on
bridge.off
bridge.emit
bridge.recv
bridge.cleanup
bridge.addCleanup
```
**plugin**
```text
bridge.plugins
bridge.unuse
bridge.use
```

**transport**
```text
bridge.chanUrl
bridge.ws
bridge.connect
bridge.send
```

correct?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes. That is the current public surface.

Two small qualifications:

- `bridge.ws` is `undefined` until `bridge.connect(chan)` succeeds; afterward it holds the active `WebSocket` or is reset to `null` when that socket closes.
- `bridge.cbs`, `bridge.cleanups`, and `bridge.plugins` are inspectable registries. Callers should normally use their corresponding lifecycle methods rather than mutate them directly.

```text
debug
  bridge.msgs

message hub & hooks lifecycle
  bridge.cbs
  bridge.cleanups
  bridge.on
  bridge.off
  bridge.emit
  bridge.recv
  bridge.cleanup
  bridge.addCleanup

plugins
  bridge.plugins
  bridge.use
  bridge.unuse

transport
  bridge.chanUrl
  bridge.ws
  bridge.connect
  bridge.send
```

`bridge.sync` and `bridge.htmx` are not generic core API: they are public APIs exposed by whichever plugins are currently active.

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 90182, "completion_tokens": 295, "total_tokens": 90477, "cached_tokens": 89472, "reasoning_tokens": 88, "cost": 0.013748550000000002}
```


And plugins use their public/internal API and the public `bridge` API, right?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Right.

A plugin’s `init(bridge)` receives the public core API and should:

- use core methods such as `bridge.on`, `bridge.off`, `bridge.recv`, `bridge.send`, and `bridge.addCleanup` where appropriate;
- keep implementation details in its closure;
- return `{api, cleanup}`;
- expose only its intentional public methods through `api`, which `bridge.use` publishes as `bridge.<name>`.

Other plugins should interact only through that exposed API, not internal closure state or registry records. For example, the HTMX plugin uses the sync plugin’s public API:

```javascript
bridge.sync?.send(el, e)
```

It should not reach into:

```javascript
bridge.plugins.sync.plugin
bridge.plugins.sync.api
```

except for inspection/debugging. That keeps plugin dependencies explicit and lets implementations change without breaking other plugins.

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 90483, "completion_tokens": 304, "total_tokens": 90787, "cached_tokens": 8576, "reasoning_tokens": 123, "cost": 0.013800449999999999}
```


Then, I think the transport public surface could be perfectly implemented with a plugin too. And thus we could be several `transport` plugins and use the plugin in other platforms. Argue with me.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

I agree. The current transport surface is the least “core” part of `bridge`:

```text
chanUrl  → assumes Solveit’s `/wsx?chan=...` endpoint
ws       → assumes WebSocket
connect  → assumes one Channel-backed socket
send     → assumes that socket is the active outbound route
```

Those are platform and transport choices, whereas the real bridge core is:

```text
local semantic message ingress  bridge.recv(...)
local event hub                 bridge.on/off/emit(...)
resource lifecycle              bridge.cleanup/addCleanup(...)
plugin lifecycle                bridge.use/unuse(...)
```

So the cleaner architecture is:

```text
bridge core
  event hub, message ingress, generic cleanup, plugin lifecycle

transport plugin
  platform-specific connection and framing
  incoming frame → bridge.recv(...)
  bridge transport events
  public send/connect/close/status API
```

For the current Solveit implementation:

```javascript
export const channelTransport = {
  init(bridge) {
    let ws = null

    const api = {
      connect: chan => { /* current WebSocket/Channel setup */ },
      send: msg => { /* current JSON send */ },
      close: () => { /* current teardown */ },
      get ws() { return ws },
    }

    return {api, cleanup:api.close}
  }
}
```

Then the bootstrap chooses it:

```javascript
bridge.use('transport', channelTransport)
bridge.transport.connect(chan)
```

`sync` becomes transport-neutral:

```javascript
bridge.transport?.send(msg)
```

and the transport plugin receives inbound parsed frames through the stable core ingress:

```javascript
bridge.recv(null, msg, event)
```

That is a strong portability boundary. An anywidget/Jupyter transport, a postMessage transport, or a test transport can each implement the same narrow contract:

```text
api.send(msg)          send one semantic JSON-compatible frame
api.connect(...)       optional platform-specific connection step
api.close()            release resources
incoming frame         calls bridge.recv(el, msg, event)
```

I would **not** support multiple active transports yet. Several transport *implementations* can be imported and selected, but a single `bridge.transport` should be active at a time. Otherwise `send` needs a routing rule—broadcast, targeted peers, fallback, acknowledgements, ordering, deduplication—which is a separate multiplexer problem.

So the public core surface would shrink to:

```text
bridge.msgs

bridge.cbs
bridge.cleanups
bridge.on
bridge.off
bridge.emit
bridge.recv
bridge.cleanup
bridge.addCleanup

bridge.plugins
bridge.use
bridge.unuse
```

And the active transport becomes plugin API rather than core API:

```text
bridge.transport
  .connect(...)
  .send(msg)
  .close()
  .ws              # only if this particular transport elects to expose it
```

That removes Solveit-specific WebSocket assumptions from the reusable bridge while preserving the current sync plugin’s semantic contract.

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 90764, "completion_tokens": 1140, "total_tokens": 91904, "cached_tokens": 89472, "reasoning_tokens": 516, "cost": 0.014469600000000003}
```


## browser bridge v2

Sounds good.Let's do it.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Start with a definition-only ESM module. This extracts the current behavior without changing the active core or event names yet.

```javascript
export const channelTransport = {
  init(bridge) {
    let ws = null
    const api = {}

    api.chanUrl = chan => {
      const url = new URL('/wsx', location.href)
      url.searchParams.set('chan', chan)
      url.protocol = location.protocol=='https:' ? 'wss:' : 'ws:'
      return url.href
    }

    api.close = () => {
      const old = ws
      ws = null
      old?.close()
    }

    api.connect = chan => {
      api.close()
      ws = new WebSocket(api.chanUrl(chan))

      ws.onopen = event => bridge.emit('channel:open', {chan, ws, event})

      ws.onmessage = event => {
        try { bridge.recv(null, JSON.parse(event.data), event) }
        catch (err) { bridge.emit('channel:error', {chan, ws, event, err}) }
      }

      ws.onerror = event => bridge.emit('channel:error', {chan, ws, event})

      ws.onclose = event => {
        if (ws===event.target) ws = null
        bridge.emit('channel:close', {chan, ws:event.target, event})
      }

      return ws
    }

    api.send = msg => {
      if (!ws || ws.readyState!=WebSocket.OPEN) return false
      ws.send(JSON.stringify(msg))
      return true
    }

    Object.defineProperty(api, 'ws', {get:() => ws})
    return {api, cleanup:api.close}
  }
}
```

This establishes the transport-plugin API:

```text
bridge.transport.chanUrl(chan)
bridge.transport.connect(chan)
bridge.transport.send(msg)
bridge.transport.close()
bridge.transport.ws
```

It deliberately preserves the existing `channel:open`, `channel:close`, and `channel:error` events during extraction.

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 89586, "completion_tokens": 1233, "total_tokens": 90819, "cached_tokens": 3456, "reasoning_tokens": 811, "cost": 0.014362650000000001}
```


In [ ]:
await add_js_msg('_channel_js', i_collapsed=True, exported=True);

In [ ]:
_channel_js = r'''export const channelTransport = {
  init(bridge) {
    let ws = null
    const api = {}

    api.chanUrl = chan => {
      const url = new URL('/wsx', location.href)
      url.searchParams.set('chan', chan)
      url.protocol = location.protocol=='https:' ? 'wss:' : 'ws:'
      return url.href
    }

    api.close = () => {
      const old = ws
      ws = null
      old?.close()
    }

    api.connect = chan => {
      api.close()
      ws = new WebSocket(api.chanUrl(chan))

      ws.onopen = event => bridge.emit('channel:open', {chan, ws, event})

      ws.onmessage = event => {
        try { bridge.recv(null, JSON.parse(event.data), event) }
        catch (err) { bridge.emit('channel:error', {chan, ws, event, err}) }
      }

      ws.onerror = event => bridge.emit('channel:error', {chan, ws, event})

      ws.onclose = event => {
        if (ws===event.target) ws = null
        bridge.emit('channel:close', {chan, ws:event.target, event})
      }

      return ws
    }

    api.send = msg => {
      if (!ws || ws.readyState!=WebSocket.OPEN) return false
      ws.send(JSON.stringify(msg))
      return true
    }

    Object.defineProperty(api, 'ws', {get:() => ws})
    return {api, cleanup:api.close}
  }
}'''
#| linkedto: 08be7925

In [ ]:
await setup_mod('channel', _channel_js, True)

('channel_1',
 {'data_id': '8ae55bb9-ee59-441b-b41f-7a88eb12f703',
  'result': {'channelTransport': {}}})

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Activate the imported transport definition without changing the existing core connection yet:

```python
r = await js_eval_a(r"""
const {channelTransport} = await import('/esm/channel?v=1')
bridge.use('transport', channelTransport)

return {
  active: bridge.plugins.transport.plugin===channelTransport,
  api: ['chanUrl','connect','send','close','ws'].every(k => k in bridge.transport),
  ws: bridge.transport.ws===null,
}
""")

test_eq(r.result, dict(active=True, api=True, ws=True))
```

This proves the plugin lifecycle and public transport API before moving the active connection from legacy `bridge.connect(...)` to `bridge.transport.connect(...)`.

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 92655, "completion_tokens": 378, "total_tokens": 93033, "cached_tokens": 3456, "reasoning_tokens": 232, "cost": 0.014181750000000002}
```


In [ ]:
r = await js_eval_a(r"""
const {channelTransport} = await import('/esm/channel?v=1')
bridge.use('transport', channelTransport)

return {
  active: bridge.plugins.transport.plugin===channelTransport,
  api: ['chanUrl','connect','send','close','ws'].every(k => k in bridge.transport),
  ws: bridge.transport.ws===null,
}
""")

test_eq(r.result, dict(active=True, api=True, ws=True))

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The transport plugin is active but intentionally disconnected. Before moving the live connection, make `sync` transport-neutral: it should use the active transport API rather than legacy `bridge.send`.

Change this one line in the `sync` module:

```javascript
bridge.transport?.send(msg)
```

in place of:

```javascript
bridge.send(msg)
```

The next step can then replace the active sync instance, close the legacy core connection, and connect `bridge.transport` to `chan` as one migration boundary.

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 92984, "completion_tokens": 625, "total_tokens": 93609, "cached_tokens": 92544, "reasoning_tokens": 516, "cost": 0.014416350000000001}
```


In [ ]:
await add_js_msg('_sync_js', i_collapsed=True, exported=True);

In [ ]:
_sync_js = r'''export const syncPlugin = {
  init(bridge) {
    const api = {}

    api.val = el => {
      if (el instanceof HTMLInputElement) {
        if (el.type=='checkbox') return el.checked
        if (['range', 'number'].includes(el.type)) {
          const v = el.valueAsNumber
          return Number.isNaN(v) ? null : v
        }
        return el.value
      }
      if (el instanceof HTMLSelectElement) return el.multiple ? [...el.selectedOptions].map(o => o.value) : el.value
      if (el instanceof HTMLTextAreaElement) return el.value
    }

    api.send = (el, event) => {
      const path = el.getAttribute('hx-sync'), value = api.val(el)
      if (!path || value===undefined) return
      const msg = {key:'sync', op:'set', path, value, raw:el.value}
      bridge.recv(el, msg, event)
      bridge.transport?.send(msg)
      return msg
    }

    api.apply = msg => {
      if (msg.key!='sync' || msg.op!='set') return
      for (const el of document.querySelectorAll('[hx-sync]')) {
        if (el.getAttribute('hx-sync')!=msg.path) continue
        if (el instanceof HTMLInputElement && el.type=='checkbox') el.checked = msg.value
        else if (el instanceof HTMLSelectElement && el.multiple) {
          for (const o of el.options) o.selected = msg.value.includes(o.value)
        } else if (el instanceof HTMLInputElement && el.type=='number' && msg.value==null) el.value = ''
        else if (el instanceof HTMLInputElement || el instanceof HTMLSelectElement || el instanceof HTMLTextAreaElement) el.value = msg.value
      }
    }

    const change = e => api.send(e.target, e)
    document.addEventListener('input', change)
    document.addEventListener('change', change)
    const off = bridge.on('msg', ({msg}) => api.apply(msg))

    return {api, cleanup: () => {
      document.removeEventListener('input', change)
      document.removeEventListener('change', change)
      off()
    }}
  }
}'''
#| linkedto: 36953ffc

In [ ]:
await setup_mod('sync', _sync_js, True)

('sync_4',
 {'data_id': 'd2d5ee27-79af-4768-b117-389150afad7d',
  'result': {'syncPlugin': {}}})

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Migrate the active connection in one boundary: replace `sync` with its transport-neutral definition, close the legacy core socket, then connect the transport plugin to the same channel.

```python
await js_eval_a(f"""
const {{syncPlugin}} = await import('/esm/sync?v=4')
bridge.use('sync', syncPlugin)
bridge.cleanup('channel')
bridge.transport.connect('{chan}')
""")

await wait_for_a(lambda: hx.eval('bridge.transport.ws?.readyState')==1)
test_eq(hx.eval('bridge.transport.ws?.readyState'), 1)
```

At this point, `sync` sends through `bridge.transport.send(...)`; the legacy core socket has been closed.

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 94412, "completion_tokens": 663, "total_tokens": 95075, "cached_tokens": 8576, "reasoning_tokens": 516, "cost": 0.01465905}
```


In [ ]:
await js_eval_a(f"""
const {{syncPlugin}} = await import('/esm/sync?v=4')
bridge.use('sync', syncPlugin)
bridge.cleanup('channel')
bridge.transport.connect('{chan}')
""")

await wait_for_a(lambda: hx.eval('bridge.transport.ws?.readyState')==1)
test_eq(hx.eval('bridge.transport.ws?.readyState'), 1)

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The Channel transport is now connected through the plugin. Re-run the browser → kernel path to verify `sync` uses `bridge.transport.send(...)` rather than the legacy core socket:

```python
state.title = 'before transport migration'

await js_eval_a(r"""
bridge.msgs = []
const el = document.getElementById('sync-title')
el.value = 'through transport plugin'
el.dispatchEvent(new Event('input', {bubbles:true}))
""")

await wait_for_a(lambda: state.title=='through transport plugin')
test_eq(state.title, 'through transport plugin')
```

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 94700, "completion_tokens": 229, "total_tokens": 94929, "cached_tokens": 3456, "reasoning_tokens": 106, "cost": 0.01437675}
```


In [ ]:
state.title = 'before transport migration'

await js_eval_a(r"""
bridge.msgs = []
const el = document.getElementById('sync-title')
el.value = 'through transport plugin'
el.dispatchEvent(new Event('input', {bubbles:true}))
""")

await wait_for_a(lambda: state.title=='through transport plugin')
test_eq(state.title, 'through transport plugin')

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Browser → kernel delivery now passes through the transport plugin. Test the reverse route: `set_sync` sends through Python `Channel`, then the transport plugin must deliver it to `sync.apply`.

```python
await set_sync('state.title', 'from kernel transport')

await wait_for_a(lambda: hx.eval("$('#sync-title')[0].value")=='from kernel transport')
test_eq((state.title, await hx.eval_a("$('#sync-title')[0].value")), ('from kernel transport', 'from kernel transport'))
```

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 94966, "completion_tokens": 284, "total_tokens": 95250, "cached_tokens": 8576, "reasoning_tokens": 173, "cost": 0.014457900000000001}
```


In [ ]:
await set_sync('state.title', 'from kernel transport')

await wait_for_a(lambda: hx.eval("$('#sync-title')[0].value")=='from kernel transport')
test_eq((state.title, await hx.eval_a("$('#sync-title')[0].value")), ('from kernel transport', 'from kernel transport'))

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Both directions now pass through `bridge.transport`. Test the transport-plugin lifecycle itself: replacing it must close its active socket and expose a fresh, disconnected API.

```python
r = await js_eval_a(r"""
const {channelTransport} = await import('/esm/channel?v=1')
const old = bridge.transport.ws

const closed = new Promise(resolve => {
  const off = bridge.on('channel:close', ({ws}) => {
    if (ws===old) {
      off()
      resolve()
    }
  })
})

bridge.use('transport', channelTransport)
await closed

return {old:old.readyState, current:bridge.transport.ws===null}
""")

test_eq(r.result, dict(old=3, current=True))
```

Then reconnect the replacement instance:

```python
await js_eval_a(f"""bridge.transport.connect('{chan}')""")
await wait_for_a(lambda: hx.eval('bridge.transport.ws?.readyState')==1)
test_eq(hx.eval('bridge.transport.ws?.readyState'), 1)
```

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 95205, "completion_tokens": 994, "total_tokens": 96199, "cached_tokens": 94592, "reasoning_tokens": 778, "cost": 0.015026250000000001}
```


In [ ]:
r = await js_eval_a(r"""
const {channelTransport} = await import('/esm/channel?v=1')
const old = bridge.transport.ws

const closed = new Promise(resolve => {
  const off = bridge.on('channel:close', ({ws}) => {
    if (ws===old) {
      off()
      resolve()
    }
  })
})

bridge.use('transport', channelTransport)
await closed

return {old:old.readyState, current:bridge.transport.ws===null}
""")

test_eq(r.result, dict(old=3, current=True))

In [ ]:
await js_eval_a(f"""bridge.transport.connect('{chan}')""")
await wait_for_a(lambda: hx.eval('bridge.transport.ws?.readyState')==1)
test_eq(hx.eval('bridge.transport.ws?.readyState'), 1)

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The transport extraction and replacement lifecycle pass. Remove the legacy transport surface from the core module now.

Delete the `bridge.chanUrl`, `bridge.connect`, and `bridge.send` definitions. Add this migration cleanup after the plugin manager:

```javascript
delete bridge.chanUrl
delete bridge.ws
delete bridge.connect
delete bridge.send
```

That ensures reloading the ESM core also removes stale properties left on `globalThis.bridge` by earlier versions.

The resulting core contains only message, cleanup, and plugin lifecycle. Transport is exclusively:

```javascript
bridge.transport.chanUrl(chan)
bridge.transport.connect(chan)
bridge.transport.send(msg)
bridge.transport.close()
bridge.transport.ws
```

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 95685, "completion_tokens": 466, "total_tokens": 96151, "cached_tokens": 8576, "reasoning_tokens": 321, "cost": 0.014702250000000002}
```


In [ ]:
await add_js_msg('_bridge_js', i_collapsed=True, exported=True);

In [ ]:
_bridge_js = r'''const bridge = globalThis.bridge || {}

bridge.msgs ||= []
bridge.cbs ||= {}
bridge.cleanups ||= {}

bridge.on = (nm, f) => {
  ;(bridge.cbs[nm] ||= []).push(f)
  return () => bridge.off(nm, f)
}

bridge.off = (nm, f) => {
  if (!f) return delete bridge.cbs[nm]
  bridge.cbs[nm] = (bridge.cbs[nm] || []).filter(g => g!==f)
  if (!bridge.cbs[nm].length) delete bridge.cbs[nm]
}

bridge.emit = (nm, data) => (bridge.cbs[nm] || []).map(f => f(data))

bridge.recv = (el, msg, event) => {
  bridge.msgs.push(msg)
  bridge.emit('msg', {el, msg, event})
  return msg
}

bridge.cleanup = nm => {
  const f = bridge.cleanups[nm]
  delete bridge.cleanups[nm]
  return f?.()
}

bridge.addCleanup = (nm, f) => {
  bridge.cleanup(nm)
  bridge.cleanups[nm] = f
  return () => bridge.cleanup(nm)
}

bridge.plugins ||= {}
delete bridge.addPlugin

bridge.unuse = nm => {
  const p = bridge.plugins[nm]
  if (!p) return
  p.cleanup()
  delete bridge.plugins[nm]
  if (bridge[nm]===p.api) delete bridge[nm]
}

bridge.use = (nm, plugin) => {
  bridge.unuse(nm)
  const {api={},cleanup=()=>{}} = plugin.init(bridge) || {}
  bridge.plugins[nm] = {plugin,api,cleanup}
  bridge[nm] = api
  return api
}

delete bridge.chanUrl
delete bridge.ws
delete bridge.connect
delete bridge.send

globalThis.bridge = bridge
export {bridge}'''
#| linkedto: 2c3b5c0a

In [ ]:
await setup_mod('bridge', _bridge_js, True)

('bridge_3',
 {'data_id': '0689ce18-b219-4e96-ba54-bdc1843411a0',
  'result': {'bridge': {'msgs': [{'key': 'sync', 'op': 'set', 'path': 'state.title', 'value': 'through transport plugin', 'raw': 'through transport plugin'}, {'key': 'sync', 'op': 'set', 'path': 'state.title', 'value': 'from kernel transport'}],
    'cbs': {'msg': [None]},
    'cleanups': {},
    'plugins': {'htmx': {'plugin': {}, 'api': {}},
     'sync': {'plugin': {}, 'api': {}},
     'transport': {'plugin': {}, 'api': {}}},
    'htmx': {},
    'sync': {},
    'transport': {}}}})

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The core reload removed the legacy transport fields while preserving the active transport plugin. Confirm both sides of that boundary:

```python
r = await js_eval_a(r"""
return {
  legacy: ['chanUrl','ws','connect','send'].every(k => bridge[k]===undefined),
  active: ['htmx','sync','transport'].every(k => k in bridge.plugins),
  open: bridge.transport.ws?.readyState==WebSocket.OPEN,
}
""")

test_eq(r.result, dict(legacy=True, active=True, open=True))
```

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 96892, "completion_tokens": 296, "total_tokens": 97188, "cached_tokens": 8576, "reasoning_tokens": 180, "cost": 0.014755800000000001}
```


In [ ]:
r = await js_eval_a(r"""
return {
  legacy: ['chanUrl','ws','connect','send'].every(k => bridge[k]===undefined),
  active: ['htmx','sync','transport'].every(k => k in bridge.plugins),
  open: bridge.transport.ws?.readyState==WebSocket.OPEN,
}
""")

test_eq(r.result, dict(legacy=True, active=True, open=True))

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The boundary is clean. Run one final post-refactor smoke test: an input event must still reach the kernel with no legacy `bridge.send` available.

```python
state.title = 'before final smoke test'

await js_eval_a(r"""
const el = document.getElementById('sync-title')
el.value = 'after final core reload'
el.dispatchEvent(new Event('input', {bubbles:true}))
""")

await wait_for_a(lambda: state.title=='after final core reload')
test_eq(state.title, 'after final core reload')
```

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 97159, "completion_tokens": 234, "total_tokens": 97393, "cached_tokens": 8576, "reasoning_tokens": 117, "cost": 0.014749350000000003}
```


In [ ]:
state.title = 'before final smoke test'

await js_eval_a(r"""
const el = document.getElementById('sync-title')
el.value = 'after final core reload'
el.dispatchEvent(new Event('input', {bubbles:true}))
""")

await wait_for_a(lambda: state.title=='after final core reload')
test_eq(state.title, 'after final core reload')

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The transport extraction is complete.

```text
bridge core
  message ingress, local event hub, named cleanup, plugin lifecycle

bridge.transport
  Channel/WebSocket connection, JSON send/receive, connection lifecycle
```

The final smoke test confirms `sync` still delivers browser changes to the kernel after legacy `bridge.send`, `bridge.connect`, `bridge.ws`, and `bridge.chanUrl` were removed.

The active plugin APIs are now:

```text
bridge.sync
bridge.htmx
bridge.transport
```

with `bridge.transport` providing:

```text
chanUrl, connect, send, close, ws
```

The core is now transport-neutral; another platform can supply a different transport plugin without changing `sync` or the bridge lifecycle machinery.

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 97418, "completion_tokens": 220, "total_tokens": 97638, "cached_tokens": 96640, "reasoning_tokens": 59, "cost": 0.014777700000000001}
```


Great! What's next?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Next: turn the control-composition exploration into the `sync` plugin, beginning with **inherited leaf paths only**.

First scope:

```text
<div hx-sync="state">
  <input hx-sync="count">
</div>
```

should make the existing browser event emit:

```text
path: "state.count"
```

without yet adding root snapshots or changing kernel assignment rules.

The small implementation sequence is:

1. Add `bridge.sync.path(el)`, caching the resolved dotted path in HTMX element data.
2. Change `sync.send` to use `api.path(el)` rather than the leaf’s literal `hx-sync`.
3. Change leaf `sync.apply` matching to compare resolved paths.
4. Re-run flat-control tests, then add nested-scope and duplicate-leaf-name tests.

This is the right next boundary because transport is now independent and stable, while full-path resolution is needed before whole-`state` snapshots or composed durable projections can be made real.

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 97638, "completion_tokens": 723, "total_tokens": 98361, "cached_tokens": 8576, "reasoning_tokens": 516, "cost": 0.015187950000000002}
```


# checkpoint

Agreed, but before that I believe we should checkpoint here and compact `## browser bridge v1` and `## browser bridge v2`. I can do it myself and come back to you to check everything's correct. Deal?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Deal. Compacting first is sensible: the v1/v2 refactors established clean boundaries, and preserving their contracts and tests in a concise record will make the inherited-path work easier to reason about.

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 97945, "completion_tokens": 44, "total_tokens": 97989, "cached_tokens": 8576, "cost": 0.014724750000000002}
```


# export -

In [ ]:
#|hide
#|eval: false
from pote.dutil import ctxusage, strip_ctx_usage
await ctxusage()

18767

In [ ]:
#|hide
#|eval: false
from pote.flakes import show_flakes
await show_flakes()

No warnings to report

In [ ]:
#|hide
#|eval: false
from pote.dialog import dlg_export
await dlg_export()